# Stage 4B — Injected User-Day Reconstruction

## Step 4B.1 — Source File & Schema Audit

**Objective:** Inspect the approved input datasets before reconstructing
the injected User-Day table.

**Expected Output:**
- 4,000 users.
- 1,394,010 user-day records.
- Original columns preserved.
- Only approved synthetic injection counts added to targeted records.

**Input Datasets:**
1. Original User-Day dataset.
2. Approved Channel-Day Ground Truth.
3. Final injected behavioral features (validation reference).

**Safety:** Read-only inspection. No source files are modified or overwritten.

**Next Step:** Reconstruct the injected activity using the approved
Formula B results.


In [5]:
# ============================================================
# STEP 4B.1 — SOURCE FILE & SCHEMA AUDIT
# READ ONLY — NO FILE MODIFICATIONS
# ============================================================

from pathlib import Path
import duckdb

PROJECT_ROOT = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)

CLEAR_DIR = PROJECT_ROOT / "Clear_Attack_Simulation"

ORIGINAL_USER_DAY = (
    PROJECT_ROOT
    / "Data_Engineering"
    / "Data_engineering"
    / "data"
    / "processed"
    / "user_day_r62_v1.parquet"
)

GT_CHANNEL_DAYS = (
    CLEAR_DIR
    / "data"
    / "ground_truth"
    / "ground_truth_channel_days.parquet"
)

REFERENCE_FEATURES = (
    CLEAR_DIR
    / "data"
    / "behavior_features_injected_final.parquet"
)

OUTPUT_PATH = (
    CLEAR_DIR
    / "data"
    / "user_day_injected_r62.parquet"
)

inputs = {
    "Original User-Day": ORIGINAL_USER_DAY,
    "Ground Truth Channel-Days": GT_CHANNEL_DAYS,
    "Final Injected Features": REFERENCE_FEATURES
}

# ------------------------------------------------------------
# 1. Check exact file paths
# ------------------------------------------------------------

print("=" * 65)
print("STEP 4B.1 — SOURCE FILE AUDIT")
print("=" * 65)

missing = []

for name, path in inputs.items():

    print(f"\n{name}")
    print("Path:", path)
    print("Exists:", path.is_file())

    if not path.is_file():
        missing.append(name)

if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}\n"
        "Check the exact paths printed above."
    )

# ------------------------------------------------------------
# 2. Inspect datasets
# ------------------------------------------------------------

con = duckdb.connect(database=":memory:")
results = {}

try:
    for name, path in inputs.items():

        sql_path = path.as_posix().replace("'", "''")

        rows, users = con.execute(f"""
            SELECT
                COUNT(*),
                COUNT(DISTINCT user)
            FROM read_parquet('{sql_path}')
        """).fetchone()

        schema = con.execute(f"""
            DESCRIBE
            SELECT *
            FROM read_parquet('{sql_path}')
        """).fetchall()

        columns = [row[0] for row in schema]

        results[name] = {
            "rows": rows,
            "users": users,
            "columns": columns
        }

        print("\n" + "=" * 65)
        print(name)
        print("=" * 65)
        print("Rows:", f"{rows:,}")
        print("Users:", f"{users:,}")
        print("Columns:", len(columns))
        print("Column names:", columns)

    # --------------------------------------------------------
    # 3. Validate expected dimensions
    # --------------------------------------------------------

    assert results["Original User-Day"]["rows"] == 1_394_010
    assert results["Original User-Day"]["users"] == 4_000

    assert results["Ground Truth Channel-Days"]["rows"] == 14_375
    assert results["Ground Truth Channel-Days"]["users"] == 670

    assert results["Final Injected Features"]["rows"] == 69_634
    assert results["Final Injected Features"]["users"] == 670

    # --------------------------------------------------------
    # 4. Check required columns
    # --------------------------------------------------------

    required_daily = {
        "user", "day",
        "external_email_count",
        "email_send_count",
        "email_event_count",
        "usb_copy_count",
        "file_copy_count",
        "file_event_count",
        "www_upload_count",
        "http_event_count"
    }

    required_gt = {
        "user", "day", "channel",
        "natural_count",
        "injected_count",
        "final_count"
    }

    assert required_daily.issubset(
        results["Original User-Day"]["columns"]
    ), "Missing original activity columns."

    assert required_gt.issubset(
        results["Ground Truth Channel-Days"]["columns"]
    ), "Missing Ground Truth columns."

    print("\n" + "=" * 65)
    print("PASS — ALL SOURCE FILES VERIFIED")
    print("No files were modified.")
    print("Planned output:", OUTPUT_PATH)
    print("=" * 65)

finally:
    con.close()

STEP 4B.1 — SOURCE FILE AUDIT

Original User-Day
Path: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\data\processed\user_day_r62_v1.parquet
Exists: True

Ground Truth Channel-Days
Path: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Clear_Attack_Simulation\data\ground_truth\ground_truth_channel_days.parquet
Exists: True

Final Injected Features
Path: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Clear_Attack_Simulation\data\behavior_features_injected_final.parquet
Exists: True

Original User-Day
Rows: 1,394,010
Users: 4,000
Columns: 34
Column names: ['user', 'day', 'day_of_week', 'is_weekend', 'email_event_count', 'email_send_count', 'email_view_count', 'external_email_count', 'internal_email_count', 'total_sent_email_bytes', 'external_email_bytes', 'emails_with_attachments', 'attachment_count', 'unique_external_recipients', 'unique_external_domains', 'file_event_count', 'file_copy_count', 'file_open_count', 'file_write_count', 'file_delete_count'

## Step 4B.2 — Injection Mapping & Consistency Validation

### Objective
Validate the approved synthetic injection records against the original
User-Day dataset before applying any modifications.

### Validation Checks
1. Verify that each User-Day record has a unique `(user, day)` key.
2. Verify that each injection has a unique `(user, day, channel)` key.
3. Confirm that all injected User-Day records exist in the original dataset.
4. Confirm that natural activity counts match the original User-Day values.
5. Verify that `final_count = natural_count + injected_count`.
6. Summarize injection counts by Email, USB, and HTTP.

### Injection Mapping
- Email → `external_email_count`
- USB → `usb_copy_count`
- HTTP → `www_upload_count`

### Safety
Read-only validation. No source files are modified or new datasets saved.

### Expected Result
All checks must pass before reconstructing the injected User-Day dataset.

In [6]:
# ============================================================
# STEP 4B.2 — INJECTION MAPPING & CONSISTENCY VALIDATION
# READ ONLY — NO FILE MODIFICATIONS
# ============================================================

import duckdb

con = duckdb.connect(database=":memory:")

def sql_path(path):
    return path.as_posix().replace("'", "''")

try:
    # --------------------------------------------------------
    # 1. Register original datasets as read-only views
    # --------------------------------------------------------

    con.execute(f"""
        CREATE VIEW original_days AS
        SELECT *
        FROM read_parquet('{sql_path(ORIGINAL_USER_DAY)}')
    """)

    con.execute(f"""
        CREATE VIEW injections AS
        SELECT *
        FROM read_parquet('{sql_path(GT_CHANNEL_DAYS)}')
    """)

    # --------------------------------------------------------
    # 2. Check unique keys and missing values
    # --------------------------------------------------------

    original_issues = con.execute("""
        SELECT
            COUNT(*) - COUNT(DISTINCT (user, day))
                AS duplicate_user_days,

            COUNT(*) FILTER (
                WHERE user IS NULL OR day IS NULL
            ) AS null_keys

        FROM original_days
    """).fetchone()

    injection_issues = con.execute("""
        SELECT
            COUNT(*) - COUNT(
                DISTINCT (user, day, LOWER(TRIM(channel)))
            ) AS duplicate_injections,

            COUNT(*) FILTER (
                WHERE user IS NULL
                   OR day IS NULL
                   OR channel IS NULL
            ) AS null_keys,

            COUNT(*) FILTER (
                WHERE LOWER(TRIM(channel))
                      NOT IN ('email', 'usb', 'http')
            ) AS unknown_channels,

            COUNT(*) FILTER (
                WHERE injected_count IS NULL
                   OR injected_count <= 0
            ) AS invalid_injected_counts

        FROM injections
    """).fetchone()

    # --------------------------------------------------------
    # 3. Match every injection against original activity
    # --------------------------------------------------------

    validation = con.execute("""
        WITH matched AS (
            SELECT
                g.user,
                g.day,
                g.channel,
                g.natural_count,
                g.injected_count,
                g.final_count,

                o.user AS matched_user,

                CASE LOWER(TRIM(g.channel))
                    WHEN 'email'
                        THEN o.external_email_count
                    WHEN 'usb'
                        THEN o.usb_copy_count
                    WHEN 'http'
                        THEN o.www_upload_count
                END AS original_count

            FROM injections g

            LEFT JOIN original_days o
              ON g.user = o.user
             AND g.day = o.day
        )

        SELECT
            COUNT(*) AS injection_rows,

            COUNT(*) FILTER (
                WHERE matched_user IS NULL
            ) AS unmatched_rows,

            COUNT(*) FILTER (
                WHERE matched_user IS NOT NULL
                  AND natural_count
                      IS DISTINCT FROM original_count
            ) AS natural_count_mismatches,

            COUNT(*) FILTER (
                WHERE final_count IS DISTINCT FROM
                      natural_count + injected_count
            ) AS final_count_mismatches

        FROM matched
    """).fetchone()

    # --------------------------------------------------------
    # 4. Summarize injections by channel
    # --------------------------------------------------------

    channel_summary = con.execute("""
        SELECT
            LOWER(TRIM(channel)) AS channel,
            COUNT(*) AS injection_rows,
            COUNT(DISTINCT user) AS users,
            SUM(injected_count) AS injected_events

        FROM injections

        GROUP BY 1
        ORDER BY 1
    """).df()

    # --------------------------------------------------------
    # 5. Display validation results
    # --------------------------------------------------------

    print("=" * 65)
    print("STEP 4B.2 — INJECTION VALIDATION")
    print("=" * 65)

    print("\nORIGINAL USER-DAY")
    print("Duplicate user-days:", original_issues[0])
    print("Null keys:", original_issues[1])

    print("\nGROUND TRUTH")
    print("Duplicate injections:", injection_issues[0])
    print("Null keys:", injection_issues[1])
    print("Unknown channels:", injection_issues[2])
    print("Invalid injected counts:", injection_issues[3])

    print("\nINJECTION MATCHING")
    print("Injection rows:", f"{validation[0]:,}")
    print("Unmatched rows:", validation[1])
    print("Natural count mismatches:", validation[2])
    print("Final count mismatches:", validation[3])

    print("\nINJECTION SUMMARY BY CHANNEL")
    print(channel_summary.to_string(index=False))

    # --------------------------------------------------------
    # 6. Stop if any validation check fails
    # --------------------------------------------------------

    checks = {
        "Original duplicate keys": original_issues[0],
        "Original null keys": original_issues[1],
        "Injection duplicate keys": injection_issues[0],
        "Injection null keys": injection_issues[1],
        "Unknown channels": injection_issues[2],
        "Invalid injected counts": injection_issues[3],
        "Unmatched injections": validation[1],
        "Natural count mismatches": validation[2],
        "Final count mismatches": validation[3],
    }

    failed = {
        name: value
        for name, value in checks.items()
        if value != 0
    }

    assert validation[0] == 14_375, (
        "Unexpected injection row count."
    )

    assert set(channel_summary["channel"]) == {
        "email", "usb", "http"
    }, "Unexpected channel values."

    assert not failed, f"Validation failed: {failed}"

    print("\n" + "=" * 65)
    print("PASS — ALL INJECTION MAPPINGS VERIFIED")
    print("No files were modified.")
    print("=" * 65)

finally:
    con.close()

STEP 4B.2 — INJECTION VALIDATION

ORIGINAL USER-DAY
Duplicate user-days: 0
Null keys: 0

GROUND TRUTH
Duplicate injections: 0
Null keys: 0
Unknown channels: 0
Invalid injected counts: 0

INJECTION MATCHING
Injection rows: 14,375
Unmatched rows: 0
Natural count mismatches: 0
Final count mismatches: 0

INJECTION SUMMARY BY CHANNEL
channel  injection_rows  users  injected_events
  email            9343    670          23880.0
   http            1533    338           6793.0
    usb            3499    499          11422.0

PASS — ALL INJECTION MAPPINGS VERIFIED
No files were modified.


## Step 4B.3 — Validate Affected Columns

### Objective
Identify and validate all User-Day columns affected by the
approved synthetic injections.

### Injection Rules

**Email**
- external_email_count
- email_send_count
- email_event_count

**USB**
- usb_copy_count
- file_copy_count
- file_event_count

**HTTP**
- www_upload_count
- http_event_count

Each affected count must increase by the approved injected count
for the corresponding user, day, and channel.

### Validation
1. Aggregate the approved injections by user and day.
2. Reconstruct the expected values of all 34 original columns.
3. Compare these values against the approved final feature dataset.
4. Ensure all unaffected columns retain their original values.

### Safety
Read-only validation. No datasets will be created or modified.

### Expected Result
Zero mismatches across all compared columns.

In [7]:
# ============================================================
# STEP 4B.3 — VALIDATE AFFECTED USER-DAY COLUMNS
# READ ONLY — NO FILE MODIFICATIONS
# ============================================================

import duckdb

con = duckdb.connect(database=":memory:")

def safe_path(path):
    return path.as_posix().replace("'", "''")

try:

    # --------------------------------------------------------
    # 1. Load approved datasets as views
    # --------------------------------------------------------

    con.execute(f"""
        CREATE VIEW original_days AS
        SELECT *
        FROM read_parquet('{safe_path(ORIGINAL_USER_DAY)}')
    """)

    con.execute(f"""
        CREATE VIEW injections AS
        SELECT *
        FROM read_parquet('{safe_path(GT_CHANNEL_DAYS)}')
    """)

    con.execute(f"""
        CREATE VIEW final_features AS
        SELECT *
        FROM read_parquet('{safe_path(REFERENCE_FEATURES)}')
    """)

    # --------------------------------------------------------
    # 2. Aggregate approved injections by User-Day
    # --------------------------------------------------------

    con.execute("""
        CREATE VIEW daily_injections AS
        SELECT
            user,
            day,

            SUM(
                CASE WHEN LOWER(TRIM(channel)) = 'email'
                THEN injected_count ELSE 0 END
            ) AS email_delta,

            SUM(
                CASE WHEN LOWER(TRIM(channel)) = 'usb'
                THEN injected_count ELSE 0 END
            ) AS usb_delta,

            SUM(
                CASE WHEN LOWER(TRIM(channel)) = 'http'
                THEN injected_count ELSE 0 END
            ) AS http_delta

        FROM injections
        GROUP BY user, day
    """)

    # --------------------------------------------------------
    # 3. Define affected columns
    # --------------------------------------------------------

    affected_columns = {
        "external_email_count": "email_delta",
        "email_send_count": "email_delta",
        "email_event_count": "email_delta",

        "usb_copy_count": "usb_delta",
        "file_copy_count": "usb_delta",
        "file_event_count": "usb_delta",

        "www_upload_count": "http_delta",
        "http_event_count": "http_delta"
    }

    original_columns = [
        row[0]
        for row in con.execute("""
            DESCRIBE SELECT * FROM original_days
        """).fetchall()
    ]

    reference_columns = {
        row[0]
        for row in con.execute("""
            DESCRIBE SELECT * FROM final_features
        """).fetchall()
    }

    assert len(original_columns) == 34
    assert set(original_columns).issubset(reference_columns)

    assert set(affected_columns).issubset(
        set(original_columns)
    )

    # --------------------------------------------------------
    # 4. Check reference keys and coverage
    # --------------------------------------------------------

    reference_checks = con.execute("""
        SELECT
            COUNT(*) AS rows,
            COUNT(*) - COUNT(DISTINCT (user, day))
                AS duplicate_keys,
            COUNT(*) FILTER (
                WHERE user IS NULL OR day IS NULL
            ) AS null_keys
        FROM final_features
    """).fetchone()

    missing_original = con.execute("""
        SELECT COUNT(*)
        FROM final_features f
        LEFT JOIN original_days o
          ON f.user = o.user
         AND f.day = o.day
        WHERE o.user IS NULL
    """).fetchone()[0]

    missing_injections = con.execute("""
        SELECT COUNT(*)
        FROM daily_injections d
        LEFT JOIN final_features f
          ON d.user = f.user
         AND d.day = f.day
        WHERE f.user IS NULL
    """).fetchone()[0]

    assert reference_checks[0] == 69_634
    assert reference_checks[1] == 0
    assert reference_checks[2] == 0
    assert missing_original == 0
    assert missing_injections == 0

    # --------------------------------------------------------
    # 5. Independently calculate expected column values
    # --------------------------------------------------------

    comparison_expressions = []

    for column in original_columns:

        quoted = f'"{column}"'

        expected = f'o.{quoted}'

        if column in affected_columns:
            delta = affected_columns[column]
            expected = (
                f'o.{quoted} + COALESCE(d.{delta}, 0)'
            )

        comparison_expressions.append(f"""
            COUNT(*) FILTER (
                WHERE f.{quoted}
                IS DISTINCT FROM ({expected})
            ) AS "{column}"
        """)

    comparison_sql = f"""
        SELECT
            COUNT(*) AS compared_rows,
            {",".join(comparison_expressions)}
        FROM final_features f

        JOIN original_days o
          ON f.user = o.user
         AND f.day = o.day

        LEFT JOIN daily_injections d
          ON f.user = d.user
         AND f.day = d.day
    """

    cursor = con.execute(comparison_sql)

    values = cursor.fetchone()

    names = [
        item[0]
        for item in cursor.description
    ]

    audit = dict(zip(names, values))

    # --------------------------------------------------------
    # 6. Display results
    # --------------------------------------------------------

    print("=" * 65)
    print("STEP 4B.3 — AFFECTED COLUMN VALIDATION")
    print("=" * 65)

    print("\nReference rows:", f"{audit['compared_rows']:,}")
    print("Original columns checked:", len(original_columns))
    print("Affected columns:", len(affected_columns))
    print("Unchanged columns:", len(original_columns) - len(affected_columns))

    print("\nAFFECTED COLUMN RESULTS")

    for column in affected_columns:
        print(f"{column}: {audit[column]} mismatches")

    unchanged_columns = [
        column
        for column in original_columns
        if column not in affected_columns
    ]

    unchanged_mismatches = sum(
        audit[column]
        for column in unchanged_columns
    )

    print("\nUNCHANGED COLUMN RESULTS")
    print("Columns checked:", len(unchanged_columns))
    print("Total mismatches:", unchanged_mismatches)

    mismatches = {
        column: audit[column]
        for column in original_columns
        if audit[column] != 0
    }

    assert audit["compared_rows"] == 69_634, (
        "Unexpected reference row count."
    )

    assert not mismatches, (
        f"Column validation failed: {mismatches}"
    )

    print("\n" + "=" * 65)
    print("PASS — ALL 34 ORIGINAL COLUMNS VERIFIED")
    print("No files were modified.")
    print("=" * 65)

finally:
    con.close()

STEP 4B.3 — AFFECTED COLUMN VALIDATION

Reference rows: 69,634
Original columns checked: 34
Affected columns: 8
Unchanged columns: 26

AFFECTED COLUMN RESULTS
external_email_count: 0 mismatches
email_send_count: 0 mismatches
email_event_count: 0 mismatches
usb_copy_count: 0 mismatches
file_copy_count: 0 mismatches
file_event_count: 0 mismatches
www_upload_count: 0 mismatches
http_event_count: 0 mismatches

UNCHANGED COLUMN RESULTS
Columns checked: 26
Total mismatches: 0

PASS — ALL 34 ORIGINAL COLUMNS VERIFIED
No files were modified.


## Step 4B.4 — Reconstruct and Save Injected User-Day

### Objective
Reconstruct the complete User-Day dataset using the approved
synthetic injection records.

### Method
1. Start from the original 4,000-user User-Day dataset.
2. Aggregate approved injections by user and day.
3. Apply injection deltas to the eight affected activity columns.
4. Preserve all other original columns and records.
5. Validate the reconstructed data against the approved features.
6. Save the result as a new Parquet file.

### Expected Output
- 1,394,010 rows
- 4,000 users
- 34 original columns
- 42,095 injected events
- Zero unexplained differences

### Safety
- Original Data Engineering files remain unchanged.
- Approved Ground Truth and Features remain unchanged.
- No existing output file will be overwritten.
- Output is saved only after validation passes.

In [8]:
# ============================================================
# STEP 4B.4 — RECONSTRUCT INJECTED USER-DAY
# SAVE ONLY AFTER SUCCESSFUL VALIDATION
# ============================================================

from pathlib import Path
import duckdb
import uuid

# Paths are defined in Step 4B.1:
# ORIGINAL_USER_DAY, GT_CHANNEL_DAYS,
# REFERENCE_FEATURES, OUTPUT_PATH

def sql_file(path):
    return path.as_posix().replace("'", "''")

# ------------------------------------------------------------
# 1. Protect existing files
# ------------------------------------------------------------

assert ORIGINAL_USER_DAY.is_file()
assert GT_CHANNEL_DAYS.is_file()
assert REFERENCE_FEATURES.is_file()

assert not OUTPUT_PATH.exists(), (
    f"Output already exists. Nothing will be overwritten:\n"
    f"{OUTPUT_PATH}"
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Temporary output, unique to this execution
temp_path = OUTPUT_PATH.with_name(
    f"{OUTPUT_PATH.stem}_building_{uuid.uuid4().hex[:8]}.parquet"
)

con = duckdb.connect(database=":memory:")

try:
    # --------------------------------------------------------
    # 2. Register approved source datasets
    # --------------------------------------------------------

    con.execute(f"""
        CREATE VIEW original_days AS
        SELECT *
        FROM read_parquet('{sql_file(ORIGINAL_USER_DAY)}')
    """)

    con.execute(f"""
        CREATE VIEW injections AS
        SELECT *
        FROM read_parquet('{sql_file(GT_CHANNEL_DAYS)}')
    """)

    con.execute(f"""
        CREATE VIEW reference_features AS
        SELECT *
        FROM read_parquet('{sql_file(REFERENCE_FEATURES)}')
    """)

    # --------------------------------------------------------
    # 3. Validate injection values
    # --------------------------------------------------------

    injection_check = con.execute("""
        SELECT
            COUNT(*),
            SUM(injected_count),
            COUNT(*) FILTER (
                WHERE injected_count IS NULL
                   OR injected_count <= 0
                   OR injected_count <> FLOOR(injected_count)
            )
        FROM injections
    """).fetchone()

    assert injection_check[0] == 14_375
    assert injection_check[1] == 42_095
    assert injection_check[2] == 0

    # --------------------------------------------------------
    # 4. Aggregate injected events by user and day
    # --------------------------------------------------------

    con.execute("""
        CREATE VIEW daily_delta AS
        SELECT
            user,
            day,

            SUM(
                CASE WHEN LOWER(TRIM(channel)) = 'email'
                THEN CAST(injected_count AS BIGINT)
                ELSE 0 END
            ) AS email_delta,

            SUM(
                CASE WHEN LOWER(TRIM(channel)) = 'usb'
                THEN CAST(injected_count AS BIGINT)
                ELSE 0 END
            ) AS usb_delta,

            SUM(
                CASE WHEN LOWER(TRIM(channel)) = 'http'
                THEN CAST(injected_count AS BIGINT)
                ELSE 0 END
            ) AS http_delta

        FROM injections
        GROUP BY user, day
    """)

    # --------------------------------------------------------
    # 5. Define the eight affected columns
    # --------------------------------------------------------

    affected = {
        "external_email_count": "email_delta",
        "email_send_count": "email_delta",
        "email_event_count": "email_delta",

        "usb_copy_count": "usb_delta",
        "file_copy_count": "usb_delta",
        "file_event_count": "usb_delta",

        "www_upload_count": "http_delta",
        "http_event_count": "http_delta"
    }

    original_schema = con.execute("""
        DESCRIBE SELECT * FROM original_days
    """).fetchall()

    original_columns = [item[0] for item in original_schema]

    assert len(original_columns) == 34
    assert set(affected).issubset(original_columns)

    # Preserve original column order and data types
    select_columns = []

    for column, data_type, *_ in original_schema:

        quoted = '"' + column.replace('"', '""') + '"'

        if column in affected:
            delta = affected[column]

            expression = (
                f'CAST(o.{quoted} + '
                f'COALESCE(d.{delta}, 0) AS {data_type}) '
                f'AS {quoted}'
            )
        else:
            expression = f'o.{quoted}'

        select_columns.append(expression)

    select_sql = ",\n".join(select_columns)

    # --------------------------------------------------------
    # 6. Reconstruct complete User-Day
    # --------------------------------------------------------

    con.execute(f"""
        CREATE VIEW reconstructed_days AS

        SELECT
            {select_sql}

        FROM original_days o

        LEFT JOIN daily_delta d
          ON o.user = d.user
         AND o.day = d.day
    """)

    print("=" * 65)
    print("STEP 4B.4 — RECONSTRUCT INJECTED USER-DAY")
    print("=" * 65)

    print("\nReconstruction view created.")

    # --------------------------------------------------------
    # 7. Write temporary Parquet for validation
    # --------------------------------------------------------

    con.execute(f"""
        COPY (
            SELECT * FROM reconstructed_days
        )
        TO '{sql_file(temp_path)}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    con.execute(f"""
        CREATE VIEW saved_days AS
        SELECT *
        FROM read_parquet('{sql_file(temp_path)}')
    """)

    # --------------------------------------------------------
    # 8. Validate dimensions and original schema
    # --------------------------------------------------------

    rows, users, unique_keys = con.execute("""
        SELECT
            COUNT(*),
            COUNT(DISTINCT user),
            COUNT(DISTINCT (user, day))
        FROM saved_days
    """).fetchone()

    saved_schema = con.execute("""
        DESCRIBE SELECT * FROM saved_days
    """).fetchall()

    saved_columns = [item[0] for item in saved_schema]

    assert rows == 1_394_010
    assert users == 4_000
    assert unique_keys == rows
    assert saved_columns == original_columns

    assert [
        (item[0], item[1]) for item in saved_schema
    ] == [
        (item[0], item[1]) for item in original_schema
    ], "Original column types changed."

    print("\nDATASET DIMENSIONS")
    print("Rows:", f"{rows:,}")
    print("Users:", f"{users:,}")
    print("Columns:", len(saved_columns))
    print("Duplicate User-Day keys:", rows - unique_keys)

    # --------------------------------------------------------
    # 9. Check all reconstructed values
    # --------------------------------------------------------

    comparisons = []

    for column in original_columns:

        quoted = '"' + column.replace('"', '""') + '"'

        expected = f'o.{quoted}'

        if column in affected:
            delta = affected[column]
            expected = (
                f'o.{quoted} + COALESCE(d.{delta}, 0)'
            )

        comparisons.append(
            f's.{quoted} IS DISTINCT FROM ({expected})'
        )

    mismatch_condition = " OR\n".join(comparisons)

    reconstruction_audit = con.execute(f"""
        SELECT
            COUNT(*) AS matched_rows,

            COUNT(*) FILTER (
                WHERE {mismatch_condition}
            ) AS mismatched_rows

        FROM saved_days s

        JOIN original_days o
          ON s.user = o.user
         AND s.day = o.day

        LEFT JOIN daily_delta d
          ON s.user = d.user
         AND s.day = d.day
    """).fetchone()

    assert reconstruction_audit[0] == 1_394_010
    assert reconstruction_audit[1] == 0

    print("\nFULL DATASET VALIDATION")
    print("Compared rows:", f"{reconstruction_audit[0]:,}")
    print("Mismatched rows:", reconstruction_audit[1])

    # --------------------------------------------------------
    # 10. Compare against approved 115-feature dataset
    # --------------------------------------------------------

    reference_conditions = [
        f's."{column}" IS DISTINCT FROM f."{column}"'
        for column in original_columns
    ]

    reference_condition = " OR\n".join(reference_conditions)

    reference_audit = con.execute(f"""
        SELECT
            COUNT(*) AS compared_rows,

            COUNT(*) FILTER (
                WHERE {reference_condition}
            ) AS mismatched_rows

        FROM reference_features f

        JOIN saved_days s
          ON f.user = s.user
         AND f.day = s.day
    """).fetchone()

    assert reference_audit[0] == 69_634
    assert reference_audit[1] == 0

    print("\nAPPROVED FEATURE REFERENCE")
    print("Compared rows:", f"{reference_audit[0]:,}")
    print("Mismatched rows:", reference_audit[1])

    # --------------------------------------------------------
    # 11. Validate injected event totals
    # --------------------------------------------------------

    totals = con.execute("""
        SELECT
            SUM(
                s.external_email_count
                - o.external_email_count
            ) AS email_added,

            SUM(
                s.usb_copy_count
                - o.usb_copy_count
            ) AS usb_added,

            SUM(
                s.www_upload_count
                - o.www_upload_count
            ) AS http_added

        FROM saved_days s

        JOIN original_days o
          ON s.user = o.user
         AND s.day = o.day
    """).fetchone()

    assert tuple(int(x) for x in totals) == (
        23_880, 11_422, 6_793
    )

    print("\nVERIFIED INJECTED EVENTS")
    print("Email:", f"{int(totals[0]):,}")
    print("USB:", f"{int(totals[1]):,}")
    print("HTTP:", f"{int(totals[2]):,}")
    print("Total:", f"{sum(int(x) for x in totals):,}")

    # --------------------------------------------------------
    # 12. Publish validated output
    # --------------------------------------------------------

    assert not OUTPUT_PATH.exists(), (
        "Final output path became occupied."
    )

    temp_path.rename(OUTPUT_PATH)

    print("\n" + "=" * 65)
    print("PASS — INJECTED USER-DAY SAVED AND VERIFIED")
    print("Final output:", OUTPUT_PATH)
    print("Original files remain unchanged.")
    print("=" * 65)

finally:
    con.close()

    # Only clean up this execution's temporary output.
    # Existing project files are never deleted.
    if temp_path.exists():
        temp_path.unlink()
        

STEP 4B.4 — RECONSTRUCT INJECTED USER-DAY

Reconstruction view created.

DATASET DIMENSIONS
Rows: 1,394,010
Users: 4,000
Columns: 34
Duplicate User-Day keys: 0

FULL DATASET VALIDATION
Compared rows: 1,394,010
Mismatched rows: 0

APPROVED FEATURE REFERENCE
Compared rows: 69,634
Mismatched rows: 0

VERIFIED INJECTED EVENTS
Email: 23,880
USB: 11,422
HTTP: 6,793
Total: 42,095

PASS — INJECTED USER-DAY SAVED AND VERIFIED
Final output: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Clear_Attack_Simulation\data\user_day_injected_r62.parquet
Original files remain unchanged.


## Step 05.1 — Source Files & Coverage Audit

### Objective
Verify the source datasets required to construct the final
Machine Learning feature dataset.

### Input Datasets
1. Original Behavioral Features (85 columns)
2. Approved Injected Features (115 columns)
3. Approved Ground Truth User-Days

### Validation Checks
- Confirm file existence and dataset dimensions.
- Verify unique (user, day) keys.
- Check the 85 original columns against the injected dataset.
- Verify coverage of injected users and their days.
- Verify Ground Truth coverage for attack-day labeling.

### Planned Output
behavior_features_r62_ml_final.parquet

Expected structure:
- 85 original columns
- 30 additional behavioral features
- 1 binary label (y)

### Safety
Read-only audit. No files will be created or modified.


In [10]:
# ============================================================
# STEP 05.1 — SOURCE FILES & COVERAGE AUDIT
# READ ONLY — NO FILE MODIFICATIONS
# ============================================================

from pathlib import Path
import duckdb

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)

CLEAR_DIR = PROJECT_ROOT / "Clear_Attack_Simulation"

ORIGINAL_FEATURES = (
    PROJECT_ROOT
    / "Data_Engineering"
    / "Data_engineering"
    / "data"
    / "features"
    / "behavior_features_r62_v1.parquet"
)

INJECTED_FEATURES = (
    CLEAR_DIR
    / "data"
    / "behavior_features_injected_final.parquet"
)

FINAL_OUTPUT = (
    CLEAR_DIR
    / "data"
    / "behavior_features_r62_ml_final.parquet"
)

# Locate approved Ground Truth without assuming its subfolder
gt_matches = sorted(
    (CLEAR_DIR / "data").rglob(
        "ground_truth_user_days.parquet"
    )
)

if len(gt_matches) != 1:
    raise FileNotFoundError(
        "Expected one ground_truth_user_days.parquet "
        f"inside Clear data; found {len(gt_matches)}.\n"
        f"Matches: {gt_matches}"
    )

GROUND_TRUTH = gt_matches[0]

sources = {
    "Original Features": ORIGINAL_FEATURES,
    "Injected Features": INJECTED_FEATURES,
    "Ground Truth User-Days": GROUND_TRUTH
}

def sql_path(path):
    return path.as_posix().replace("'", "''")

# ------------------------------------------------------------
# 2. Verify source paths
# ------------------------------------------------------------

print("=" * 65)
print("STEP 05.1 — SOURCE FILES & COVERAGE AUDIT")
print("=" * 65)

for name, path in sources.items():
    print(f"\n{name}")
    print("Path:", path)
    print("Exists:", path.is_file())

assert all(path.is_file() for path in sources.values()), (
    "One or more required source files are missing."
)

assert not FINAL_OUTPUT.exists(), (
    "Final output already exists. "
    "Do not overwrite it without reviewing."
)

# ------------------------------------------------------------
# 3. Read schema and dimensions
# ------------------------------------------------------------

con = duckdb.connect(database=":memory:")

try:
    for view_name, path in [
        ("original_features", ORIGINAL_FEATURES),
        ("injected_features", INJECTED_FEATURES),
        ("ground_truth", GROUND_TRUTH)
    ]:
        con.execute(f"""
            CREATE VIEW {view_name} AS
            SELECT *
            FROM read_parquet('{sql_path(path)}')
        """)

    datasets = {}

    for display_name, view_name in [
        ("Original Features", "original_features"),
        ("Injected Features", "injected_features"),
        ("Ground Truth", "ground_truth")
    ]:

        rows, users, unique_keys, null_keys = con.execute(f"""
            SELECT
                COUNT(*),
                COUNT(DISTINCT user),
                COUNT(DISTINCT (user, day)),
                COUNT(*) FILTER (
                    WHERE user IS NULL OR day IS NULL
                )
            FROM {view_name}
        """).fetchone()

        columns = [
            item[0]
            for item in con.execute(
                f"DESCRIBE SELECT * FROM {view_name}"
            ).fetchall()
        ]

        datasets[display_name] = {
            "rows": rows,
            "users": users,
            "unique_keys": unique_keys,
            "null_keys": null_keys,
            "columns": columns
        }

        print("\n" + "=" * 65)
        print(display_name)
        print("=" * 65)

        print("Rows:", f"{rows:,}")
        print("Users:", f"{users:,}")
        print("Columns:", len(columns))
        print("Duplicate keys:", rows - unique_keys)
        print("Null keys:", null_keys)

    # --------------------------------------------------------
    # 4. Verify basic dimensions
    # --------------------------------------------------------

    original = datasets["Original Features"]
    injected = datasets["Injected Features"]
    gt = datasets["Ground Truth"]

    assert original["rows"] == 387_162
    assert original["users"] == 3_737
    assert len(original["columns"]) == 85

    assert injected["rows"] == 69_634
    assert injected["users"] == 670
    assert len(injected["columns"]) == 115

    assert gt["rows"] == 69_634
    assert gt["users"] == 670
    assert "is_attack_day" in gt["columns"]

    for name, info in datasets.items():
        assert info["unique_keys"] == info["rows"], (
            f"Duplicate keys found in {name}"
        )
        assert info["null_keys"] == 0, (
            f"Null keys found in {name}"
        )

    # --------------------------------------------------------
    # 5. Verify feature column compatibility
    # --------------------------------------------------------

    original_cols = original["columns"]
    injected_cols = injected["columns"]

    missing_original_columns = (
        set(original_cols) - set(injected_cols)
    )

    new_feature_columns = [
        col for col in injected_cols
        if col not in original_cols
    ]

    print("\n" + "=" * 65)
    print("FEATURE SCHEMA COMPATIBILITY")
    print("=" * 65)

    print(
        "Missing original columns:",
        len(missing_original_columns)
    )
    print(
        "Additional behavioral columns:",
        len(new_feature_columns)
    )

    assert not missing_original_columns
    assert len(new_feature_columns) == 30

    # --------------------------------------------------------
    # 6. Check injected row coverage
    # --------------------------------------------------------

    coverage = con.execute("""
        WITH original_users AS (
            SELECT DISTINCT user
            FROM original_features
        ),

        missing_in_original AS (
            SELECT i.user, i.day
            FROM injected_features i
            ANTI JOIN original_features o
              ON i.user = o.user
             AND i.day = o.day
        ),

        missing_in_injected AS (
            SELECT o.user, o.day
            FROM original_features o
            SEMI JOIN (
                SELECT DISTINCT user
                FROM injected_features
            ) u
              ON o.user = u.user
            ANTI JOIN injected_features i
              ON o.user = i.user
             AND o.day = i.day
        )

        SELECT
            (
                SELECT COUNT(*)
                FROM missing_in_original
            ) AS injected_keys_missing_from_original,

            (
                SELECT COUNT(*)
                FROM missing_in_injected
            ) AS original_keys_missing_from_injected,

            (
                SELECT COUNT(*)
                FROM (
                    SELECT DISTINCT user
                    FROM injected_features
                ) i
                ANTI JOIN original_users o
                  ON i.user = o.user
            ) AS injected_users_missing_from_original
    """).fetchone()

    print("\n" + "=" * 65)
    print("INJECTED FEATURES COVERAGE")
    print("=" * 65)

    print(
        "Injected keys missing from original:",
        coverage[0]
    )
    print(
        "Original keys missing from injected:",
        coverage[1]
    )
    print(
        "Injected users missing from original:",
        coverage[2]
    )

    # --------------------------------------------------------
    # 7. Check Ground Truth coverage and binary label
    # --------------------------------------------------------

    gt_checks = con.execute("""
        SELECT
            (
                SELECT COUNT(*)
                FROM injected_features i
                ANTI JOIN ground_truth g
                  ON i.user = g.user
                 AND i.day = g.day
            ) AS features_without_gt,

            (
                SELECT COUNT(*)
                FROM ground_truth g
                ANTI JOIN injected_features i
                  ON g.user = i.user
                 AND g.day = i.day
            ) AS gt_without_features,

            (
                SELECT COUNT(*)
                FROM ground_truth
                WHERE is_attack_day IS NULL
                   OR is_attack_day NOT IN (0, 1)
            ) AS invalid_labels,

            (
                SELECT COUNT(*)
                FROM ground_truth
                WHERE is_attack_day = 1
            ) AS positive_days
    """).fetchone()

    print("\n" + "=" * 65)
    print("GROUND TRUTH COVERAGE")
    print("=" * 65)

    print("Features without Ground Truth:", gt_checks[0])
    print("Ground Truth without Features:", gt_checks[1])
    print("Invalid binary labels:", gt_checks[2])
    print("Positive attack days:", f"{gt_checks[3]:,}")

    # --------------------------------------------------------
    # 8. Display final audit verdict
    # --------------------------------------------------------

    issues = {
        "Injected keys missing from original": coverage[0],
        "Original keys missing from injected": coverage[1],
        "Injected users missing from original": coverage[2],
        "Features without Ground Truth": gt_checks[0],
        "Ground Truth without Features": gt_checks[1],
        "Invalid binary labels": gt_checks[2]
    }

    failed = {
        key: value
        for key, value in issues.items()
        if value != 0
    }

    print("\n" + "=" * 65)

    if failed:
        print("REVIEW REQUIRED — COVERAGE DIFFERENCES FOUND")
        for name, value in failed.items():
            print(f"{name}: {value}")
    else:
        print("PASS — ALL SOURCE FILES & KEYS VERIFIED")

    print("No files were modified.")
    print("Planned output:", FINAL_OUTPUT)
    print("=" * 65)

finally:
    con.close()

STEP 05.1 — SOURCE FILES & COVERAGE AUDIT

Original Features
Path: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\data\features\behavior_features_r62_v1.parquet
Exists: True

Injected Features
Path: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Clear_Attack_Simulation\data\behavior_features_injected_final.parquet
Exists: True

Ground Truth User-Days
Path: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Clear_Attack_Simulation\data\ground_truth\ground_truth_user_days.parquet
Exists: True

Original Features
Rows: 387,162
Users: 3,737
Columns: 85
Duplicate keys: 0
Null keys: 0

Injected Features
Rows: 69,634
Users: 670
Columns: 115
Duplicate keys: 0
Null keys: 0

Ground Truth
Rows: 69,634
Users: 670
Columns: 13
Duplicate keys: 0
Null keys: 0

FEATURE SCHEMA COMPATIBILITY
Missing original columns: 0
Additional behavioral columns: 30

INJECTED FEATURES COVERAGE
Injected keys missing from original: 0
Original keys missing from injected: 0
Injected users m

## Step 05.2 — Non-Injected Population & Feature Logic Audit

### Objective
Identify the non-injected population and locate the approved
calculation logic for the 30 additional behavioral features.

### Checks
1. Separate the 670 injected users from the original feature population.
2. Verify the remaining users and records.
3. Check the source columns required for the new features.
4. Inspect historical baseline availability.
5. Locate the notebooks containing the approved feature formulas.

### Expected Population
- Non-injected users: 3,067
- Non-injected records: 317,528
- Original feature columns: 85
- Additional features to calculate: 30

### Safety
Read-only audit. No source datasets will be modified.

In [11]:
# ============================================================
# STEP 05.2 — NON-INJECTED POPULATION & FEATURE LOGIC AUDIT
# READ ONLY — NO FILE MODIFICATIONS
# ============================================================

from pathlib import Path
import duckdb
import json

con = duckdb.connect(database=":memory:")

def safe_path(path):
    return path.as_posix().replace("'", "''")

try:

    # --------------------------------------------------------
    # 1. Load original and injected feature references
    # --------------------------------------------------------

    con.execute(f"""
        CREATE VIEW original_features AS
        SELECT *
        FROM read_parquet('{safe_path(ORIGINAL_FEATURES)}')
    """)

    con.execute(f"""
        CREATE VIEW injected_features AS
        SELECT *
        FROM read_parquet('{safe_path(INJECTED_FEATURES)}')
    """)

    # --------------------------------------------------------
    # 2. Identify users without synthetic injections
    # --------------------------------------------------------

    con.execute("""
        CREATE VIEW non_injected_features AS
        SELECT o.*
        FROM original_features o
        WHERE NOT EXISTS (
            SELECT 1
            FROM injected_features i
            WHERE i.user = o.user
        )
    """)

    population = con.execute("""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT user) AS users,
            COUNT(DISTINCT (user, day)) AS unique_keys,
            MIN(day) AS first_day,
            MAX(day) AS last_day
        FROM non_injected_features
    """).fetchone()

    print("=" * 65)
    print("STEP 05.2 — NON-INJECTED POPULATION AUDIT")
    print("=" * 65)

    print("\nNON-INJECTED POPULATION")
    print("Rows:", f"{population[0]:,}")
    print("Users:", f"{population[1]:,}")
    print("Unique User-Day keys:", f"{population[2]:,}")
    print("First date:", population[3])
    print("Last date:", population[4])

    assert population[0] == 317_528
    assert population[1] == 3_067
    assert population[2] == population[0]

    # --------------------------------------------------------
    # 3. Inspect required columns
    # --------------------------------------------------------

    required_columns = [
        "user",
        "day",
        "is_weekend",
        "external_email_count",
        "usb_copy_count",
        "www_upload_count",
        "email_count_p50",
        "email_count_p95",
        "usb_copy_p50",
        "usb_copy_p95",
        "http_upload_p50",
        "http_upload_p95"
    ]

    original_columns = [
        row[0]
        for row in con.execute("""
            DESCRIBE SELECT * FROM non_injected_features
        """).fetchall()
    ]

    missing = [
        column
        for column in required_columns
        if column not in original_columns
    ]

    print("\nREQUIRED SOURCE COLUMNS")
    print("Columns checked:", len(required_columns))
    print("Missing columns:", missing)

    assert not missing, (
        f"Required columns are missing: {missing}"
    )

    # --------------------------------------------------------
    # 4. Inspect baseline availability
    # --------------------------------------------------------

    baseline_columns = [
        "email_count_p50",
        "email_count_p95",
        "usb_copy_p50",
        "usb_copy_p95",
        "http_upload_p50",
        "http_upload_p95"
    ]

    print("\nHISTORICAL BASELINE AVAILABILITY")

    for column in baseline_columns:

        missing_count = con.execute(f"""
            SELECT COUNT(*)
            FROM non_injected_features
            WHERE "{column}" IS NULL
        """).fetchone()[0]

        print(
            f"{column}: "
            f"{missing_count:,} NULL rows"
        )

    # --------------------------------------------------------
    # 5. Check workday history available for rolling windows
    # --------------------------------------------------------

    workday_summary = con.execute("""
        WITH history AS (
            SELECT
                user,
                COUNT(*) AS total_rows,
                COUNT(*) FILTER (
                    WHERE is_weekend = 0
                ) AS workdays
            FROM non_injected_features
            GROUP BY user
        )

        SELECT
            COUNT(*) AS total_users,
            COUNT(*) FILTER (
                WHERE workdays >= 30
            ) AS users_with_30_workdays,
            COUNT(*) FILTER (
                WHERE workdays < 30
            ) AS users_below_30_workdays,
            MIN(workdays) AS min_workdays,
            MAX(workdays) AS max_workdays
        FROM history
    """).fetchone()

    print("\nROLLING WINDOW COVERAGE")
    print("Users:", workday_summary[0])
    print("Users with >=30 workdays:", workday_summary[1])
    print("Users with <30 workdays:", workday_summary[2])
    print("Minimum workdays:", workday_summary[3])
    print("Maximum workdays:", workday_summary[4])

finally:
    con.close()

# ------------------------------------------------------------
# 6. Locate existing notebooks containing new feature logic
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("APPROVED FEATURE LOGIC — NOTEBOOK DISCOVERY")
print("=" * 65)

search_roots = [
    CLEAR_DIR,
    PROJECT_ROOT / "Attack_Simulation"
]

feature_keywords = [
    "cumulative_deviation_",
    "combined_channel_intensity_",
    "dominant_channel_switches_",
    "channel_distribution_shift_",
    "consecutive_elevated_days_",
    "cross_channel_deviation_days_"
]

notebook_matches = []

for root in search_roots:

    if not root.exists():
        continue

    for notebook_path in root.rglob("*.ipynb"):

        try:
            notebook = json.loads(
                notebook_path.read_text(encoding="utf-8")
            )

            matching_cells = []

            for index, cell in enumerate(
                notebook.get("cells", [])
            ):

                if cell.get("cell_type") != "code":
                    continue

                source = cell.get("source", [])

                if isinstance(source, list):
                    source = "".join(source)

                hits = [
                    keyword
                    for keyword in feature_keywords
                    if keyword in source
                ]

                if hits:
                    matching_cells.append({
                        "cell": index + 1,
                        "keywords": hits
                    })

            if matching_cells:
                notebook_matches.append(
                    (notebook_path, matching_cells)
                )

        except (OSError, UnicodeError, ValueError) as error:
            print(
                f"Could not inspect {notebook_path.name}: "
                f"{type(error).__name__}"
            )

print("Notebooks with feature references:", len(notebook_matches))

for path, cells in notebook_matches:

    print("\nNotebook:", path)

    for cell in cells[:10]:
        print(
            "  Cell:",
            cell["cell"],
            "| Feature groups:",
            len(cell["keywords"])
        )

    if len(cells) > 10:
        print("  Additional matching cells:", len(cells) - 10)

print("\n" + "=" * 65)
print("PASS — NON-INJECTED POPULATION VERIFIED")
print("No files were created or modified.")
print("=" * 65)

STEP 05.2 — NON-INJECTED POPULATION AUDIT

NON-INJECTED POPULATION
Rows: 317,528
Users: 3,067
Unique User-Day keys: 317,528
First date: 2011-01-03
Last date: 2011-06-01

REQUIRED SOURCE COLUMNS
Columns checked: 12
Missing columns: []

HISTORICAL BASELINE AVAILABILITY
email_count_p50: 0 NULL rows
email_count_p95: 0 NULL rows
usb_copy_p50: 294,569 NULL rows
usb_copy_p95: 294,569 NULL rows
http_upload_p50: 293,797 NULL rows
http_upload_p95: 293,797 NULL rows

ROLLING WINDOW COVERAGE
Users: 3067
Users with >=30 workdays: 3044
Users with <30 workdays: 23
Minimum workdays: 1
Maximum workdays: 108

APPROVED FEATURE LOGIC — NOTEBOOK DISCOVERY
Could not inspect Reconstruct Injected User-Day.ipynb: JSONDecodeError
Notebooks with feature references: 2

Notebook: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Clear_Attack_Simulation\notebokes\04_attack_scheduling_and_injection.ipynb
  Cell: 81 | Feature groups: 1
  Cell: 83 | Feature groups: 1
  Cell: 85 | Feature groups: 1
  Cell: 87 | Feature g

## Step 05.3 — Historical Baseline Audit

### Objective
Verify the historical baseline required to reproduce the
30 approved behavioral features.

### Reference
The original Attack Simulation notebook, Steps 3.4.15–3.4.22.

### Checks
1. Locate user_baseline_r62_v1.csv.
2. Verify all required historical baseline columns.
3. Confirm coverage of the 3,737 eligible users.
4. Inspect missing historical values.

### Safety
Read-only validation. No datasets are modified.

In [13]:
# ============================================================
# STEP 05.3 — HISTORICAL BASELINE AUDIT
# READ ONLY
# ============================================================

from pathlib import Path
import pandas as pd
import duckdb

print("=" * 65)
print("STEP 05.3 — HISTORICAL BASELINE AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Locate baseline file
# ------------------------------------------------------------

filename = "user_baseline_r62_v1.csv"

search_roots = [
    PROJECT_ROOT / "Data_Engineering",
    PROJECT_ROOT / "Clear_Attack_Simulation",
    PROJECT_ROOT / "Attack_Simulation"
]

matches = []

for root in search_roots:
    if root.is_dir():
        matches.extend(root.rglob(filename))

matches = list(dict.fromkeys(matches))

print("\nBASELINE FILE DISCOVERY")
print("Files found:", len(matches))

for path in matches:
    print("-", path)

assert matches, "Historical baseline file not found."

# Prefer the original Data Engineering source
BASELINE_PATH = matches[0]

print("\nSelected baseline:", BASELINE_PATH)

# ------------------------------------------------------------
# 2. Read historical baseline
# ------------------------------------------------------------

baseline = pd.read_csv(BASELINE_PATH)

required = [
    "user",
    "baseline_workdays",

    "email_count_p50",
    "email_count_p95",
    "email_count_mean_active",
    "email_active_rate",
    "email_active_days",

    "usb_copy_p50",
    "usb_copy_p95",
    "usb_copy_mean_active",
    "usb_active_rate",
    "usb_active_days",

    "http_upload_p50",
    "http_upload_p95",
    "http_upload_mean_active",
    "http_upload_active_rate",
    "http_upload_active_days"
]

missing = sorted(set(required) - set(baseline.columns))

print("\nBASELINE SCHEMA")
print("Rows:", f"{len(baseline):,}")
print("Users:", f"{baseline['user'].nunique():,}")
print("Columns:", len(baseline.columns))
print("Missing required columns:", missing)

assert not missing
assert baseline["user"].is_unique
assert baseline["user"].notna().all()

# ------------------------------------------------------------
# 3. Verify eligible-user coverage
# ------------------------------------------------------------

con = duckdb.connect(database=":memory:")

try:
    con.execute(f"""
        CREATE VIEW original_features AS
        SELECT user, day
        FROM read_parquet(
            '{ORIGINAL_FEATURES.as_posix()}'
        )
    """)

    eligible_users = set(
        con.execute("""
            SELECT DISTINCT user
            FROM original_features
        """).df()["user"].astype(str)
    )

finally:
    con.close()

baseline["user"] = baseline["user"].astype(str)

baseline_users = set(baseline["user"])

missing_users = eligible_users - baseline_users

print("\nUSER COVERAGE")
print("Eligible users:", f"{len(eligible_users):,}")
print("Users missing from baseline:", len(missing_users))

assert not missing_users, (
    f"Missing baseline users: "
    f"{list(sorted(missing_users))[:10]}"
)

# ------------------------------------------------------------
# 4. Inspect historical data availability
# ------------------------------------------------------------

eligible_baseline = baseline[
    baseline["user"].isin(eligible_users)
].copy()

print("\nHISTORICAL BASELINE NULL COUNTS")

for column in required[1:]:
    print(
        f"{column}: "
        f"{eligible_baseline[column].isna().sum():,}"
    )

print("\n" + "=" * 65)
print("PASS — HISTORICAL BASELINE SOURCE VERIFIED")
print("No files were modified.")
print("=" * 65)

STEP 05.3 — HISTORICAL BASELINE AUDIT

BASELINE FILE DISCOVERY
Files found: 1
- C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\data\features\user_baseline_r62_v1.csv

Selected baseline: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\data\features\user_baseline_r62_v1.csv

BASELINE SCHEMA
Rows: 3,995
Users: 3,995
Columns: 36
Missing required columns: []

USER COVERAGE
Eligible users: 3,737
Users missing from baseline: 0

HISTORICAL BASELINE NULL COUNTS
baseline_workdays: 0
email_count_p50: 0
email_count_p95: 0
email_count_mean_active: 0
email_active_rate: 0
email_active_days: 0
usb_copy_p50: 3,016
usb_copy_p95: 3,016
usb_copy_mean_active: 3,016
usb_active_rate: 0
usb_active_days: 0
http_upload_p50: 3,170
http_upload_p95: 3,170
http_upload_mean_active: 3,170
http_upload_active_rate: 0
http_upload_active_days: 0

PASS — HISTORICAL BASELINE SOURCE VERIFIED
No files were modified.


## Step 05.4 — Calculate Non-Injected Behavioral Features

### Objective
Calculate the 30 approved behavioral features for the
3,067 non-injected users.

### Method
- Start from the original 85-column behavioral dataset.
- Exclude the 670 injected users.
- Apply the approved feature formulas from Attack Simulation.
- Use historical baseline values without modifying them.
- Use complete causal rolling windows of 5, 10, 20, and 30 eligible workdays.
- Preserve missing values where a feature cannot be calculated.

### Expected Result
- 3,067 users
- 317,528 rows
- 115 columns (85 original + 30 new)

### Safety
No files are created, modified, or overwritten.

In [14]:
# ============================================================
# STEP 05.4 — CALCULATE 30 BEHAVIORAL FEATURES
# NON-INJECTED USERS ONLY — IN MEMORY
# ============================================================

import numpy as np
import pandas as pd
import duckdb
from numpy.lib.stride_tricks import sliding_window_view

WINDOWS = [5, 10, 20, 30]

CHANNELS = {
    "Email": {
        "count": "external_email_count",
        "p50": "email_count_p50",
        "p95": "email_count_p95",
        "mean": "email_count_mean_active",
        "rate": "email_active_rate",
        "days": "email_active_days"
    },
    "USB": {
        "count": "usb_copy_count",
        "p50": "usb_copy_p50",
        "p95": "usb_copy_p95",
        "mean": "usb_copy_mean_active",
        "rate": "usb_active_rate",
        "days": "usb_active_days"
    },
    "HTTP": {
        "count": "www_upload_count",
        "p50": "http_upload_p50",
        "p95": "http_upload_p95",
        "mean": "http_upload_mean_active",
        "rate": "http_upload_active_rate",
        "days": "http_upload_active_days"
    }
}

# ------------------------------------------------------------
# 1. Load non-injected population
# ------------------------------------------------------------

con = duckdb.connect(database=":memory:")

try:
    df = con.execute(f"""
        SELECT o.*
        FROM read_parquet(
            '{ORIGINAL_FEATURES.as_posix()}'
        ) o

        ANTI JOIN (
            SELECT DISTINCT user
            FROM read_parquet(
                '{INJECTED_FEATURES.as_posix()}'
            )
        ) i
        ON o.user = i.user

        ORDER BY o.user, o.day
    """).df()

    reference_columns = [
        row[0] for row in con.execute(f"""
            DESCRIBE
            SELECT *
            FROM read_parquet(
                '{INJECTED_FEATURES.as_posix()}'
            )
        """).fetchall()]

finally:
    con.close()

df["day"] = pd.to_datetime(df["day"]).dt.normalize()
df = df.sort_values(["user", "day"]).reset_index(drop=True)

original_columns = list(df.columns)

assert len(df) == 317_528
assert df["user"].nunique() == 3_067
assert len(original_columns) == 85
assert not df.duplicated(["user", "day"]).any()
assert df["is_weekend"].eq(0).all()

# ------------------------------------------------------------
# 2. Prepare historical baseline
# ------------------------------------------------------------

baseline = pd.read_csv(BASELINE_PATH)

baseline["user"] = baseline["user"].astype(str)
df["user"] = df["user"].astype(str)

assert baseline["user"].is_unique

baseline = baseline.set_index("user")

users = df["user"].unique()

assert set(users).issubset(set(baseline.index))

user_baseline = baseline.loc[users]

# ------------------------------------------------------------
# 3. Validate channel activity and baselines
# ------------------------------------------------------------

counts = np.column_stack([
    df[spec["count"]].to_numpy(dtype=float)
    for spec in CHANNELS.values()
])

p50 = np.column_stack([
    df[spec["p50"]].to_numpy(dtype=float)
    for spec in CHANNELS.values()
])

p95 = np.column_stack([
    df[spec["p95"]].to_numpy(dtype=float)
    for spec in CHANNELS.values()
])

assert np.isfinite(counts).all()
assert (counts >= 0).all()

valid_p95 = np.isfinite(p95) & (p95 > 0)

valid_baseline = (
    valid_p95
    & np.isfinite(p50)
    & (p50 > 0)
    & (p95 >= p50)
)

assert not ((counts > 0) & ~valid_baseline).any(), (
    "Positive activity without a valid historical baseline."
)

assert valid_p95.any(axis=1).all()

# ------------------------------------------------------------
# 4. Prepare reusable causal rolling calculations
# ------------------------------------------------------------

groups = list(
    df.groupby("user", sort=False).indices.values()
)

position = (
    df.groupby("user", sort=False).cumcount().to_numpy()
    + 1
)

def rolling_feature(values, window, calculation):

    series = pd.Series(values, index=df.index)

    return series.groupby(
        df["user"], sort=False
    ).transform(
        lambda s: getattr(
            s.rolling(window, min_periods=window),
            calculation
        )()
    ).to_numpy(dtype=float)


def rolling_switches(dominant, window):

    n = len(dominant)
    result = np.zeros(n, dtype=float)

    valid_indices = np.flatnonzero(dominant >= 0)

    if len(valid_indices) >= 2:

        changes = (
            dominant[valid_indices[1:]]
            != dominant[valid_indices[:-1]]
        ).astype(np.int64)

        cumulative = np.concatenate([
            np.array([0], dtype=np.int64),
            np.cumsum(changes)
        ])

        positions = np.arange(n)

        starts = np.maximum(
            positions - window + 1, 0
        )

        left = np.searchsorted(
            valid_indices, starts, side="left"
        )

        right = np.searchsorted(
            valid_indices, positions, side="right"
        )

        has_pair = (right - left) >= 2

        result[has_pair] = (
            cumulative[right[has_pair] - 1]
            - cumulative[left[has_pair]]
        )

    result[:min(window - 1, n)] = np.nan

    return result


def longest_run(elevated, window):

    elevated = np.asarray(elevated, dtype=bool)
    n = len(elevated)

    result = np.full(n, np.nan)

    if n < window:
        return result

    windows = sliding_window_view(elevated, window)

    current = np.zeros(len(windows), dtype=np.int32)
    longest = np.zeros(len(windows), dtype=np.int32)

    for i in range(window):

        current = np.where(
            windows[:, i], current + 1, 0
        )

        longest = np.maximum(longest, current)

    result[window - 1:] = longest

    return result


# ------------------------------------------------------------
# 5. Cumulative Deviation — 4 features
# ------------------------------------------------------------

denominator = np.maximum(p95 - p50, 1.0)

deviation = np.zeros_like(counts, dtype=float)

np.divide(
    np.maximum(counts - p50, 0),
    denominator,
    out=deviation,
    where=valid_baseline
)

daily_deviation = deviation.sum(axis=1)

for w in WINDOWS:
    df[f"cumulative_deviation_{w}wd"] = (
        rolling_feature(daily_deviation, w, "sum")
    )

print("1/7 — Cumulative Deviation: DONE")

# ------------------------------------------------------------
# 6. Combined Channel Intensity — 4 features
# ------------------------------------------------------------

normalized = np.zeros_like(counts, dtype=float)

np.divide(
    counts,
    p95,
    out=normalized,
    where=valid_p95
)

daily_intensity = (
    normalized.sum(axis=1)
    / valid_p95.sum(axis=1)
)

for w in WINDOWS:
    df[f"combined_channel_intensity_{w}wd"] = (
        rolling_feature(daily_intensity, w, "mean")
    )

print("2/7 — Combined Channel Intensity: DONE")

# ------------------------------------------------------------
# 7. Dominant Channel Switches — 4 features
# ------------------------------------------------------------

dominant_matrix = np.where(
    valid_p95, normalized, -np.inf
)

maximum = dominant_matrix.max(axis=1)

tied = np.isclose(
    dominant_matrix,
    maximum[:, None],
    rtol=1e-9,
    atol=1e-12
)

unique_positive = (
    (maximum > 0)
    & (tied.sum(axis=1) == 1)
)

dominant = np.full(len(df), -1, dtype=np.int8)

dominant[unique_positive] = np.argmax(
    dominant_matrix, axis=1
)[unique_positive]

for w in WINDOWS:

    values = np.full(len(df), np.nan)

    for indices in groups:
        values[indices] = rolling_switches(
            dominant[indices], w
        )

    df[f"dominant_channel_switches_{w}wd"] = values

print("3/7 — Dominant Channel Switches: DONE")

# ------------------------------------------------------------
# 8. Historical Channel Distribution
# ------------------------------------------------------------

historical_weights = np.zeros(
    (len(users), 3), dtype=float
)

for j, spec in enumerate(CHANNELS.values()):

    b_p95 = user_baseline[spec["p95"]].to_numpy(
        dtype=float
    )

    b_mean = user_baseline[spec["mean"]].to_numpy(
        dtype=float
    )

    b_rate = user_baseline[spec["rate"]].to_numpy(
        dtype=float
    )

    b_days = user_baseline[spec["days"]].to_numpy(
        dtype=float
    )

    usable = (
        np.isfinite(b_p95)
        & (b_p95 > 0)
        & np.isfinite(b_mean)
        & (b_mean > 0)
        & np.isfinite(b_rate)
        & (b_rate > 0)
        & (b_rate <= 1)
        & np.isfinite(b_days)
        & (b_days > 0)
    )

    inactive = (
        ~usable
        & (b_days == 0)
        & ((b_rate == 0) | np.isnan(b_rate))
    )

    assert (usable | inactive).all(), (
        f"Unresolved historical baseline: "
        f"{list(CHANNELS)[j]}"
    )

    historical_weights[usable, j] = (
        b_rate[usable]
        * b_mean[usable]
        / b_p95[usable]
    )

historical_total = historical_weights.sum(axis=1)

assert (historical_total > 0).all()

historical_shares = (
    historical_weights / historical_total[:, None]
)

share_lookup = pd.DataFrame(
    historical_shares,
    index=users,
    columns=list(CHANNELS)
)

row_shares = share_lookup.loc[
    df["user"].to_numpy()
].to_numpy(dtype=float)

# ------------------------------------------------------------
# 9. Channel Distribution Shift — 4 features
# ------------------------------------------------------------

rolling_channel_sums = {}

for w in WINDOWS:

    rolling_matrix = np.column_stack([
        rolling_feature(normalized[:, j], w, "sum")
        for j in range(3)
    ])

    total = rolling_matrix.sum(axis=1)

    valid_window = (
        (position >= w)
        & np.isfinite(total)
        & (total > 0)
    )

    observed_shares = np.zeros_like(rolling_matrix)

    np.divide(
        rolling_matrix,
        total[:, None],
        out=observed_shares,
        where=valid_window[:, None]
    )

    tvd = (
        0.5
        * np.abs(observed_shares - row_shares).sum(axis=1)
    )

    result = np.full(len(df), np.nan)

    result[valid_window] = np.clip(
        tvd[valid_window], 0, 1
    )

    df[f"channel_distribution_shift_{w}wd"] = result

print("4/7 — Channel Distribution Shift: DONE")

# ------------------------------------------------------------
# 10. Consecutive Elevated Days — 4 features
# ------------------------------------------------------------

elevated = (
    valid_baseline
    & (counts > p50)
)

elevated_any = elevated.any(axis=1)

for w in WINDOWS:

    values = np.full(len(df), np.nan)

    for indices in groups:
        values[indices] = longest_run(
            elevated_any[indices], w
        )

    df[f"consecutive_elevated_days_{w}wd"] = values

print("5/7 — Consecutive Elevated Days: DONE")

# ------------------------------------------------------------
# 11. Cross-Channel Deviation Days — 4 features
# ------------------------------------------------------------

cross_channel_day = (
    elevated.sum(axis=1) >= 2
).astype(np.int8)

for w in WINDOWS:
    df[f"cross_channel_deviation_days_{w}wd"] = (
        rolling_feature(cross_channel_day, w, "sum")
    )

print("6/7 — Cross-Channel Deviation Days: DONE")

# ------------------------------------------------------------
# 12. Rolling Channel Totals — 6 features
# ------------------------------------------------------------

prefixes = [
    "external_email_count",
    "usb_copy_count",
    "http_upload_count"
]

for w in [20, 30]:
    for j, prefix in enumerate(prefixes):
        df[f"{prefix}_{w}wd"] = (
            rolling_feature(counts[:, j], w, "sum")
        )

print("7/7 — Rolling Channel Totals: DONE")

# ------------------------------------------------------------
# 13. Validate final non-injected features
# ------------------------------------------------------------

new_columns = [
    column
    for column in df.columns
    if column not in original_columns
]

assert len(new_columns) == 30
assert len(df.columns) == 115
assert set(df.columns) == set(reference_columns)

assert not df.duplicated(["user", "day"]).any()

for w in WINDOWS:

    incomplete = position < w

    window_columns = [
        column for column in new_columns
        if column.endswith(f"_{w}wd")
    ]

    for column in window_columns:
        assert df.loc[incomplete, column].isna().all(), (
            f"Incomplete window error: {column}"
        )

# Match the exact approved 115-column ordering
non_injected_features_115 = df[
    reference_columns
].copy()

assert len(non_injected_features_115) == 317_528
assert non_injected_features_115["user"].nunique() == 3_067

print("\n" + "=" * 65)
print("STEP 05.4 — NON-INJECTED FEATURES VALIDATION")
print("=" * 65)

print("Users:", non_injected_features_115["user"].nunique())
print("Rows:", len(non_injected_features_115))
print("Original columns:", len(original_columns))
print("New features:", len(new_columns))
print("Final columns:", len(non_injected_features_115.columns))
print("Duplicate keys:", non_injected_features_115.duplicated(
    ["user", "day"]
).sum())

print("\nNEW FEATURE NULL COUNTS")

for column in new_columns:
    print(
        f"{column}: "
        f"{non_injected_features_115[column].isna().sum():,}"
    )

print("\n" + "=" * 65)
print("PASS — NON-INJECTED FEATURES CALCULATED")
print("Result retained in: non_injected_features_115")
print("No files were created or modified.")
print("=" * 65)

1/7 — Cumulative Deviation: DONE
2/7 — Combined Channel Intensity: DONE
3/7 — Dominant Channel Switches: DONE
4/7 — Channel Distribution Shift: DONE
5/7 — Consecutive Elevated Days: DONE
6/7 — Cross-Channel Deviation Days: DONE
7/7 — Rolling Channel Totals: DONE

STEP 05.4 — NON-INJECTED FEATURES VALIDATION
Users: 3067
Rows: 317528
Original columns: 85
New features: 30
Final columns: 115
Duplicate keys: 0

NEW FEATURE NULL COUNTS
cumulative_deviation_5wd: 12,265
cumulative_deviation_10wd: 27,578
cumulative_deviation_20wd: 58,152
cumulative_deviation_30wd: 88,623
combined_channel_intensity_5wd: 12,265
combined_channel_intensity_10wd: 27,578
combined_channel_intensity_20wd: 58,152
combined_channel_intensity_30wd: 88,623
dominant_channel_switches_5wd: 12,265
dominant_channel_switches_10wd: 27,578
dominant_channel_switches_20wd: 58,152
dominant_channel_switches_30wd: 88,623
channel_distribution_shift_5wd: 41,580
channel_distribution_shift_10wd: 39,712
channel_distribution_shift_20wd: 60,62

## Step 05.5 — Merge Behavioral Features and Add Labels

### Objective
Combine the approved injected features with the newly calculated
non-injected features into one labeled Machine Learning dataset.

### Inputs
- Non-Injected Features: 317,528 rows, 115 columns
- Injected Features: 69,634 rows, 115 columns
- Ground Truth User-Days: 69,634 rows

### Method
1. Verify schema compatibility.
2. Assign y=0 to non-injected records.
3. Assign y from Ground Truth to injected records.
4. Combine datasets without adding or removing User-Day records.
5. Validate all labels, keys, and dimensions.

### Expected Result
- 387,162 rows
- 3,737 users
- 116 columns including y
- 12,636 positive attack-day labels
- No duplicate User-Day keys

### Safety
The merged dataset remains in memory.
No files are created or modified.

In [15]:
# ============================================================
# STEP 05.5 — MERGE FEATURES AND ADD BINARY LABELS
# IN MEMORY — NO FILE MODIFICATIONS
# ============================================================

import pandas as pd
import duckdb

print("=" * 65)
print("STEP 05.5 — FINAL ML DATASET ASSEMBLY")
print("=" * 65)

# ------------------------------------------------------------
# 1. Load approved injected features and Ground Truth
# ------------------------------------------------------------

con = duckdb.connect(database=":memory:")

try:
    injected_df = con.execute(f"""
        SELECT *
        FROM read_parquet('{INJECTED_FEATURES.as_posix()}')
    """).df()

    gt_df = con.execute(f"""
        SELECT user, day, is_attack_day
        FROM read_parquet('{GROUND_TRUTH.as_posix()}')
    """).df()

finally:
    con.close()

# ------------------------------------------------------------
# 2. Standardize user and day keys
# ------------------------------------------------------------

non_injected_df = non_injected_features_115.copy()

for data in [non_injected_df, injected_df, gt_df]:
    data["user"] = data["user"].astype(str)
    data["day"] = pd.to_datetime(
        data["day"]
    ).dt.normalize()

# ------------------------------------------------------------
# 3. Validate input schemas and keys
# ------------------------------------------------------------

assert list(non_injected_df.columns) == reference_columns
assert list(injected_df.columns) == reference_columns

assert len(non_injected_df) == 317_528
assert len(injected_df) == 69_634
assert len(gt_df) == 69_634

assert not non_injected_df.duplicated(
    ["user", "day"]
).any()

assert not injected_df.duplicated(
    ["user", "day"]
).any()

assert not gt_df.duplicated(
    ["user", "day"]
).any()

non_injected_users = set(non_injected_df["user"])
injected_users = set(injected_df["user"])

assert non_injected_users.isdisjoint(injected_users)

assert gt_df["is_attack_day"].notna().all()
assert gt_df["is_attack_day"].isin([0, 1]).all()

print("\nINPUT VALIDATION")
print("Non-injected rows:", f"{len(non_injected_df):,}")
print("Injected rows:", f"{len(injected_df):,}")
print("Ground Truth rows:", f"{len(gt_df):,}")
print("Feature columns:", len(reference_columns))

# ------------------------------------------------------------
# 4. Label non-injected users
# ------------------------------------------------------------

non_injected_df["y"] = 0

# ------------------------------------------------------------
# 5. Label injected users using approved Ground Truth
# ------------------------------------------------------------

injected_labeled = injected_df.merge(
    gt_df,
    on=["user", "day"],
    how="left",
    validate="one_to_one",
    indicator=True
)

assert injected_labeled["_merge"].eq("both").all(), (
    "Some injected records have no Ground Truth."
)

assert injected_labeled["is_attack_day"].notna().all()

injected_labeled["y"] = (
    injected_labeled["is_attack_day"].astype("int8")
)

injected_labeled = injected_labeled.drop(
    columns=["is_attack_day", "_merge"]
)

assert injected_labeled["y"].sum() == 12_636

# ------------------------------------------------------------
# 6. Combine injected and non-injected populations
# ------------------------------------------------------------

final_ml_features = pd.concat(
    [non_injected_df, injected_labeled],
    ignore_index=True
)

final_ml_features["y"] = (
    final_ml_features["y"].astype("int8")
)

final_ml_features = (
    final_ml_features
    .sort_values(["user", "day"], kind="mergesort")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 7. Verify final dataset dimensions
# ------------------------------------------------------------

total_rows = len(final_ml_features)
total_users = final_ml_features["user"].nunique()
total_columns = len(final_ml_features.columns)

duplicate_keys = final_ml_features.duplicated(
    ["user", "day"]
).sum()

assert total_rows == 387_162
assert total_users == 3_737
assert total_columns == 116
assert duplicate_keys == 0

assert list(final_ml_features.columns) == (
    reference_columns + ["y"]
)

# ------------------------------------------------------------
# 8. Verify all original User-Day keys are preserved
# ------------------------------------------------------------

con = duckdb.connect(database=":memory:")

try:
    con.register(
        "final_keys",
        final_ml_features[["user", "day"]]
    )

    con.execute(f"""
        CREATE VIEW original_keys AS
        SELECT user, day
        FROM read_parquet(
            '{ORIGINAL_FEATURES.as_posix()}'
        )
    """)

    missing_keys, extra_keys = con.execute("""
        SELECT

            (
                SELECT COUNT(*)
                FROM original_keys o
                ANTI JOIN final_keys f
                  ON o.user = f.user
                 AND o.day = f.day
            ),

            (
                SELECT COUNT(*)
                FROM final_keys f
                ANTI JOIN original_keys o
                  ON f.user = o.user
                 AND f.day = o.day
            )
    """).fetchone()

finally:
    con.close()

assert missing_keys == 0
assert extra_keys == 0

# ------------------------------------------------------------
# 9. Validate binary labels and population separation
# ------------------------------------------------------------

assert final_ml_features["y"].isin([0, 1]).all()

positive_days = int(final_ml_features["y"].sum())
negative_days = total_rows - positive_days

assert positive_days == 12_636
assert negative_days == 374_526

non_injected_positive = final_ml_features.loc[
    final_ml_features["user"].isin(non_injected_users),
    "y"
].sum()

assert non_injected_positive == 0

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL ML DATASET VALIDATION")
print("=" * 65)

print("Rows:", f"{total_rows:,}")
print("Users:", f"{total_users:,}")
print("Columns:", total_columns)
print("Duplicate User-Day keys:", duplicate_keys)
print("Missing original keys:", missing_keys)
print("Extra keys:", extra_keys)

print("\nLABEL DISTRIBUTION")
print("Benign / Non-Attack (y=0):", f"{negative_days:,}")
print("Attack (y=1):", f"{positive_days:,}")

print(
    "Attack percentage:",
    f"{positive_days / total_rows * 100:.2f}%"
)

print("\nPOPULATION")
print("Injected users:", len(injected_users))
print("Non-injected users:", len(non_injected_users))

print("\n" + "=" * 65)
print("PASS — FINAL ML DATASET ASSEMBLED AND VALIDATED")
print("Result retained in: final_ml_features")
print("No files were created or modified.")
print("=" * 65)

STEP 05.5 — FINAL ML DATASET ASSEMBLY

INPUT VALIDATION
Non-injected rows: 317,528
Injected rows: 69,634
Ground Truth rows: 69,634
Feature columns: 115

FINAL ML DATASET VALIDATION
Rows: 387,162
Users: 3,737
Columns: 116
Duplicate User-Day keys: 0
Missing original keys: 0
Extra keys: 0

LABEL DISTRIBUTION
Benign / Non-Attack (y=0): 374,526
Attack (y=1): 12,636
Attack percentage: 3.26%

POPULATION
Injected users: 670
Non-injected users: 3067

PASS — FINAL ML DATASET ASSEMBLED AND VALIDATED
Result retained in: final_ml_features
No files were created or modified.


## Step 05.6 — Save and Verify Final ML Dataset

### Objective
Save the complete labeled behavioral feature dataset for
Machine Learning and independently verify the saved file.

### Output
behavior_features_r62_ml_final.parquet

### Expected Structure
- 387,162 rows
- 3,737 users
- 116 columns
- 85 original features
- 30 additional behavioral features
- 1 binary label: y

### Validation Checks
1. Preserve all User-Day records and column names.
2. Verify unique (user, day) keys.
3. Verify label distribution.
4. Compare all saved feature values against the in-memory dataset.
5. Preserve missing feature values without imputation.

### Safety
Save to a new file under Clear_Attack_Simulation.
Never overwrite existing files or modify original sources.

In [16]:
# ============================================================
# STEP 05.6 — SAVE & VERIFY FINAL ML DATASET
# ============================================================

from pathlib import Path
import duckdb
import uuid

print("=" * 65)
print("STEP 05.6 — SAVE FINAL ML DATASET")
print("=" * 65)

# ------------------------------------------------------------
# 1. Verify final dataset before saving
# ------------------------------------------------------------

assert "final_ml_features" in globals(), (
    "Run Step 05.5 first."
)

assert len(final_ml_features) == 387_162
assert final_ml_features["user"].nunique() == 3_737
assert len(final_ml_features.columns) == 116

assert not final_ml_features.duplicated(
    ["user", "day"]
).any()

assert list(final_ml_features.columns) == (
    reference_columns + ["y"]
)

assert final_ml_features["y"].notna().all()
assert final_ml_features["y"].isin([0, 1]).all()
assert int(final_ml_features["y"].sum()) == 12_636

# ------------------------------------------------------------
# 2. Define protected output path
# ------------------------------------------------------------

FINAL_OUTPUT = (
    PROJECT_ROOT
    / "Clear_Attack_Simulation"
    / "data"
    / "behavior_features_r62_ml_final.parquet"
)

assert not FINAL_OUTPUT.exists(), (
    f"Output already exists. Will not overwrite:\n"
    f"{FINAL_OUTPUT}"
)

FINAL_OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

TEMP_OUTPUT = FINAL_OUTPUT.with_name(
    FINAL_OUTPUT.stem
    + "_building_"
    + uuid.uuid4().hex[:8]
    + ".parquet"
)

def sql_path(path):
    return path.as_posix().replace("'", "''")

# ------------------------------------------------------------
# 3. Save temporary Parquet
# ------------------------------------------------------------

con = duckdb.connect(database=":memory:")

try:
    con.register(
        "ml_in_memory",
        final_ml_features
    )

    con.execute(f"""
        COPY (
            SELECT *
            FROM ml_in_memory
            ORDER BY user, day
        )
        TO '{sql_path(TEMP_OUTPUT)}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    print("\nTemporary Parquet created.")

    # --------------------------------------------------------
    # 4. Reopen saved Parquet
    # --------------------------------------------------------

    con.execute(f"""
        CREATE VIEW saved_ml AS
        SELECT *
        FROM read_parquet('{sql_path(TEMP_OUTPUT)}')
    """)

    saved_schema = con.execute("""
        DESCRIBE SELECT * FROM saved_ml
    """).fetchall()

    saved_columns = [
        row[0] for row in saved_schema
    ]

    assert saved_columns == list(final_ml_features.columns)

    # --------------------------------------------------------
    # 5. Verify dimensions and labels
    # --------------------------------------------------------

    saved_stats = con.execute("""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT user) AS users,
            COUNT(DISTINCT (user, day)) AS unique_keys,

            COUNT(*) FILTER (
                WHERE y = 1
            ) AS positive_days,

            COUNT(*) FILTER (
                WHERE y = 0
            ) AS negative_days,

            COUNT(*) FILTER (
                WHERE y IS NULL OR y NOT IN (0, 1)
            ) AS invalid_labels

        FROM saved_ml
    """).fetchone()

    (
        saved_rows,
        saved_users,
        unique_keys,
        positive_days,
        negative_days,
        invalid_labels
    ) = saved_stats

    assert saved_rows == 387_162
    assert saved_users == 3_737
    assert unique_keys == saved_rows
    assert positive_days == 12_636
    assert negative_days == 374_526
    assert invalid_labels == 0

    print("\nSAVED DATASET")
    print("Rows:", f"{saved_rows:,}")
    print("Users:", f"{saved_users:,}")
    print("Columns:", len(saved_columns))
    print("Duplicate keys:", saved_rows - unique_keys)

    print("\nSAVED LABELS")
    print("y=0:", f"{negative_days:,}")
    print("y=1:", f"{positive_days:,}")

    # --------------------------------------------------------
    # 6. Compare all saved feature values with memory
    # --------------------------------------------------------

    compare_columns = [
        column
        for column in saved_columns
        if column not in ["user", "day"]
    ]

    mismatch_conditions = []

    for column in compare_columns:

        quoted = '"' + column.replace('"', '""') + '"'

        mismatch_conditions.append(
            f"s.{quoted} IS DISTINCT FROM m.{quoted}"
        )

    mismatch_expression = " OR\n".join(
        mismatch_conditions
    )

    comparison = con.execute(f"""
        SELECT
            COUNT(*) AS compared_rows,

            COUNT(*) FILTER (
                WHERE {mismatch_expression}
            ) AS mismatched_rows

        FROM saved_ml s

        JOIN ml_in_memory m
          ON s.user = m.user
         AND s.day = m.day
    """).fetchone()

    compared_rows, mismatched_rows = comparison

    assert compared_rows == 387_162
    assert mismatched_rows == 0

    print("\nFULL DATA VALIDATION")
    print("Compared rows:", f"{compared_rows:,}")
    print("Compared non-key columns:", len(compare_columns))
    print("Mismatched rows:", mismatched_rows)

    # --------------------------------------------------------
    # 7. Verify no missing or extra User-Day records
    # --------------------------------------------------------

    key_validation = con.execute("""
        SELECT
            (
                SELECT COUNT(*)
                FROM ml_in_memory m
                ANTI JOIN saved_ml s
                  ON m.user = s.user
                 AND m.day = s.day
            ) AS missing_keys,

            (
                SELECT COUNT(*)
                FROM saved_ml s
                ANTI JOIN ml_in_memory m
                  ON s.user = m.user
                 AND s.day = m.day
            ) AS extra_keys
    """).fetchone()

    assert key_validation == (0, 0)

    print("\nKEY VALIDATION")
    print("Missing keys:", key_validation[0])
    print("Extra keys:", key_validation[1])

finally:
    con.close()

# ------------------------------------------------------------
# 8. Publish output only after validation
# ------------------------------------------------------------

if (
    compared_rows == 387_162
    and mismatched_rows == 0
    and key_validation == (0, 0)
):
    assert not FINAL_OUTPUT.exists(), (
        "Final output appeared during validation. "
        "Nothing will be overwritten."
    )

    TEMP_OUTPUT.rename(FINAL_OUTPUT)

    print("\n" + "=" * 65)
    print("PASS — FINAL ML DATASET SAVED AND VERIFIED")
    print("Final output:", FINAL_OUTPUT)
    print(
        "File size:",
        f"{FINAL_OUTPUT.stat().st_size / (1024**2):.2f} MB"
    )
    print("Original files remain unchanged.")
    print("=" * 65)

STEP 05.6 — SAVE FINAL ML DATASET

Temporary Parquet created.

SAVED DATASET
Rows: 387,162
Users: 3,737
Columns: 116
Duplicate keys: 0

SAVED LABELS
y=0: 374,526
y=1: 12,636

FULL DATA VALIDATION
Compared rows: 387,162
Compared non-key columns: 114
Mismatched rows: 0

KEY VALIDATION
Missing keys: 0
Extra keys: 0

PASS — FINAL ML DATASET SAVED AND VERIFIED
Final output: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Clear_Attack_Simulation\data\behavior_features_r62_ml_final.parquet
File size: 16.75 MB
Original files remain unchanged.


## Step 05.7A — Independent Data Integrity & Provenance Audit

### Objective
Independently verify the final ML dataset against its original
Data Engineering, approved injected features, and Ground Truth.

### Audit Scope
1. Verify all source and output file dimensions.
2. Verify schema and unique User-Day keys.
3. Compare all 85 original columns for non-injected users.
4. Compare all 115 feature columns for injected users.
5. Independently reconstruct every binary label from Ground Truth.
6. Verify no missing, duplicated, or additional records.
7. Check basic activity-count consistency.
8. Detect potentially dangerous metadata columns.

### Expected Results
- 387,162 rows and 3,737 users
- 116 columns
- 317,528 non-injected records
- 69,634 injected-user records
- 12,636 attack-day labels
- Zero unexplained mismatches

### Safety
Read-only independent audit.
No source or final dataset files will be modified.

In [17]:
# ============================================================
# STEP 05.7A — INDEPENDENT DATA INTEGRITY AUDIT
# READ ONLY — ALL COMPARISONS FROM DISK
# ============================================================

from pathlib import Path
import duckdb

PROJECT_ROOT = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)

CLEAR_DIR = PROJECT_ROOT / "Clear_Attack_Simulation"

ORIGINAL = (
    PROJECT_ROOT / "Data_Engineering"
    / "Data_engineering" / "data" / "features"
    / "behavior_features_r62_v1.parquet"
)

INJECTED = (
    CLEAR_DIR / "data"
    / "behavior_features_injected_final.parquet"
)

GT = (
    CLEAR_DIR / "data" / "ground_truth"
    / "ground_truth_user_days.parquet"
)

FINAL = (
    CLEAR_DIR / "data"
    / "behavior_features_r62_ml_final.parquet"
)

SOURCES = {
    "Original Features": ORIGINAL,
    "Injected Features": INJECTED,
    "Ground Truth": GT,
    "Final ML Dataset": FINAL
}

def sql_path(p):
    return p.as_posix().replace("'", "''")

def quote_col(name):
    return '"' + name.replace('"', '""') + '"'

def compare_columns(con, left, right, columns, condition=""):
    """
    Compare every value across all requested columns.
    Both datasets must have unique (user, day) keys.
    """
    expressions = []

    for index, column in enumerate(columns):
        q = quote_col(column)

        expressions.append(
            f'COUNT(*) FILTER ('
            f'WHERE a.{q} IS DISTINCT FROM b.{q}'
            f') AS mismatch_{index}'
        )

    query = f"""
        SELECT
            COUNT(*) AS compared_rows,
            {", ".join(expressions)}
        FROM {left} a
        JOIN {right} b
          ON a.user = b.user
         AND a.day = b.day
        {condition}
    """

    result = con.execute(query).fetchone()
    compared_rows = result[0]

    mismatches = {
        column: value
        for column, value in zip(columns, result[1:])
        if value != 0
    }

    return compared_rows, mismatches

# ------------------------------------------------------------
# 1. File existence
# ------------------------------------------------------------

print("=" * 70)
print("STEP 05.7A — INDEPENDENT DATA INTEGRITY AUDIT")
print("=" * 70)

for name, path in SOURCES.items():
    print(f"{name}: {path.is_file()}")
    assert path.is_file(), f"Missing file: {path}"

con = duckdb.connect(database=":memory:")

try:

    # --------------------------------------------------------
    # 2. Read all sources independently from disk
    # --------------------------------------------------------

    for view, path in [
        ("original", ORIGINAL),
        ("injected", INJECTED),
        ("gt", GT),
        ("final", FINAL)
    ]:
        con.execute(f"""
            CREATE VIEW {view} AS
            SELECT *
            FROM read_parquet('{sql_path(path)}')
        """)

    expected = {
        "original": (387_162, 3_737, 85),
        "injected": (69_634, 670, 115),
        "gt": (69_634, 670, 13),
        "final": (387_162, 3_737, 116)
    }

    schemas = {}

    print("\n" + "=" * 70)
    print("1. SOURCE DIMENSIONS & KEY INTEGRITY")
    print("=" * 70)

    for view, (exp_rows, exp_users, exp_cols) in expected.items():

        stats = con.execute(f"""
            SELECT
                COUNT(*),
                COUNT(DISTINCT user),
                COUNT(DISTINCT (user, day)),
                COUNT(*) FILTER (
                    WHERE user IS NULL OR day IS NULL
                )
            FROM {view}
        """).fetchone()

        schema = con.execute(f"""
            DESCRIBE SELECT * FROM {view}
        """).fetchall()

        cols = [row[0] for row in schema]
        schemas[view] = cols

        rows, users, unique_keys, null_keys = stats

        print(
            f"\n{view}: "
            f"{rows:,} rows | "
            f"{users:,} users | "
            f"{len(cols)} columns"
        )

        print("Duplicate keys:", rows - unique_keys)
        print("Null keys:", null_keys)

        assert rows == exp_rows
        assert users == exp_users
        assert len(cols) == exp_cols
        assert unique_keys == rows
        assert null_keys == 0

    # --------------------------------------------------------
    # 3. Verify schema compatibility
    # --------------------------------------------------------

    original_cols = schemas["original"]
    injected_cols = schemas["injected"]
    final_cols = schemas["final"]

    assert set(original_cols).issubset(
        set(injected_cols)
    )

    assert final_cols == injected_cols + ["y"], (
        "Final feature schema differs from approved injection."
    )

    added_features = [
        c for c in injected_cols
        if c not in original_cols
    ]

    assert len(added_features) == 30

    print("\n" + "=" * 70)
    print("2. SCHEMA COMPATIBILITY")
    print("=" * 70)

    print("Original columns:", len(original_cols))
    print("Additional features:", len(added_features))
    print("Binary label:", "y")
    print("Final columns:", len(final_cols))

    # --------------------------------------------------------
    # 4. Verify complete original key coverage
    # --------------------------------------------------------

    missing_keys = con.execute("""
        SELECT COUNT(*)
        FROM original o
        ANTI JOIN final f
          ON o.user = f.user
         AND o.day = f.day
    """).fetchone()[0]

    extra_keys = con.execute("""
        SELECT COUNT(*)
        FROM final f
        ANTI JOIN original o
          ON f.user = o.user
         AND f.day = o.day
    """).fetchone()[0]

    print("\n" + "=" * 70)
    print("3. ORIGINAL RECORD PRESERVATION")
    print("=" * 70)

    print("Missing original keys:", missing_keys)
    print("Unexpected final keys:", extra_keys)

    assert missing_keys == 0
    assert extra_keys == 0

    # --------------------------------------------------------
    # 5. Independently verify the non-injected population
    #
    # Final values must match all 85 original columns.
    # --------------------------------------------------------

    non_injected_rows, non_injected_mismatches = (
        compare_columns(
            con,
            "final",
            "original",
            original_cols,
            condition="""
                WHERE NOT EXISTS (
                    SELECT 1
                    FROM injected i
                    WHERE i.user = a.user
                )
            """
        )
    )

    print("\n" + "=" * 70)
    print("4. NON-INJECTED USER VERIFICATION")
    print("=" * 70)

    print("Compared rows:", f"{non_injected_rows:,}")
    print("Compared columns:", len(original_cols))
    print(
        "Columns with mismatches:",
        len(non_injected_mismatches)
    )
    print("Mismatch details:", non_injected_mismatches)

    assert non_injected_rows == 317_528
    assert not non_injected_mismatches, (
        "Non-injected original features were modified."
    )

    # --------------------------------------------------------
    # 6. Independently verify the injected population
    #
    # Final values must match all 115 approved features.
    # --------------------------------------------------------

    injected_rows, injected_mismatches = (
        compare_columns(
            con,
            "final",
            "injected",
            injected_cols
        )
    )

    print("\n" + "=" * 70)
    print("5. INJECTED USER VERIFICATION")
    print("=" * 70)

    print("Compared rows:", f"{injected_rows:,}")
    print("Compared columns:", len(injected_cols))
    print(
        "Columns with mismatches:",
        len(injected_mismatches)
    )
    print("Mismatch details:", injected_mismatches)

    assert injected_rows == 69_634
    assert not injected_mismatches, (
        "Approved injected features do not match."
    )

    # --------------------------------------------------------
    # 7. Verify complete Ground Truth coverage
    # --------------------------------------------------------

    gt_missing = con.execute("""
        SELECT COUNT(*)
        FROM gt g
        ANTI JOIN final f
          ON g.user = f.user
         AND g.day = f.day
    """).fetchone()[0]

    injected_without_gt = con.execute("""
        SELECT COUNT(*)
        FROM injected i
        ANTI JOIN gt g
          ON i.user = g.user
         AND i.day = g.day
    """).fetchone()[0]

    assert gt_missing == 0
    assert injected_without_gt == 0

    # --------------------------------------------------------
    # 8. Reconstruct EVERY label from Ground Truth
    #
    # y = Ground Truth label if present, else 0.
    # --------------------------------------------------------

    label_audit = con.execute("""
        SELECT
            COUNT(*) AS rows,

            COUNT(*) FILTER (
                WHERE f.y IS DISTINCT FROM
                    COALESCE(g.is_attack_day, 0)
            ) AS incorrect_labels,

            COUNT(*) FILTER (
                WHERE f.y IS NULL
                   OR f.y NOT IN (0, 1)
            ) AS invalid_labels,

            COUNT(*) FILTER (
                WHERE f.y = 1
            ) AS positive_days,

            COUNT(*) FILTER (
                WHERE f.y = 0
            ) AS negative_days,

            COUNT(*) FILTER (
                WHERE g.user IS NULL AND f.y = 1
            ) AS attacks_without_ground_truth

        FROM final f

        LEFT JOIN gt g
          ON f.user = g.user
         AND f.day = g.day
    """).fetchone()

    (
        label_rows,
        incorrect_labels,
        invalid_labels,
        positive_days,
        negative_days,
        attacks_without_gt
    ) = label_audit

    print("\n" + "=" * 70)
    print("6. INDEPENDENT LABEL VERIFICATION")
    print("=" * 70)

    print("Labels checked:", f"{label_rows:,}")
    print("Incorrect labels:", incorrect_labels)
    print("Invalid labels:", invalid_labels)
    print("Attack days:", f"{positive_days:,}")
    print("Non-attack days:", f"{negative_days:,}")
    print("Attacks without Ground Truth:", attacks_without_gt)
    print("Missing Ground Truth records:", gt_missing)

    assert label_rows == 387_162
    assert incorrect_labels == 0
    assert invalid_labels == 0
    assert positive_days == 12_636
    assert negative_days == 374_526
    assert attacks_without_gt == 0

    # --------------------------------------------------------
    # 9. Check logical activity count relationships
    # --------------------------------------------------------

    consistency = con.execute("""
        SELECT
            COUNT(*) FILTER (
                WHERE external_email_count > email_send_count
                   OR email_send_count > email_event_count
            ) AS email_issues,

            COUNT(*) FILTER (
                WHERE usb_copy_count > file_copy_count
                   OR file_copy_count > file_event_count
            ) AS usb_issues,

            COUNT(*) FILTER (
                WHERE www_upload_count > http_event_count
            ) AS http_issues,

            COUNT(*) FILTER (
                WHERE external_email_count < 0
                   OR email_send_count < 0
                   OR email_event_count < 0
                   OR usb_copy_count < 0
                   OR file_copy_count < 0
                   OR file_event_count < 0
                   OR www_upload_count < 0
                   OR http_event_count < 0
            ) AS negative_activity_rows

        FROM final
    """).fetchone()

    print("\n" + "=" * 70)
    print("7. ACTIVITY CONSISTENCY")
    print("=" * 70)

    print("Email count issues:", consistency[0])
    print("USB count issues:", consistency[1])
    print("HTTP count issues:", consistency[2])
    print("Negative activity rows:", consistency[3])

    # Do not automatically discard records if anomalies exist.
    # Their origin must be investigated first.

    # --------------------------------------------------------
    # 10. Scan for explicit attack metadata leakage
    # --------------------------------------------------------

    forbidden = [
        "scenario_id",
        "stealth_level",
        "is_attack_day",
        "injected_count",
        "injection_formula",
        "attack_start",
        "attack_end",
        "ground_truth_source",
        "planned_switches"
    ]

    leaked_columns = [
        c for c in final_cols
        if c in forbidden
    ]

    print("\n" + "=" * 70)
    print("8. ATTACK METADATA LEAKAGE SCREEN")
    print("=" * 70)

    print("Forbidden metadata columns:", leaked_columns)

    assert not leaked_columns, (
        "Explicit attack metadata found in ML features."
    )

    # --------------------------------------------------------
    # 11. Final verdict
    # --------------------------------------------------------

    print("\n" + "=" * 70)

    if any(value != 0 for value in consistency):
        print("REVIEW REQUIRED — ACTIVITY ANOMALIES FOUND")
        print(
            "Determine whether these anomalies already "
            "exist in the original source data."
        )
    else:
        print("PASS — INDEPENDENT DATA INTEGRITY AUDIT")

    print("\nSource integrity: VERIFIED")
    print("Feature provenance: VERIFIED")
    print("Ground Truth labels: VERIFIED")
    print("Record preservation: VERIFIED")
    print("Feature formulas: NOT YET INDEPENDENTLY RECOMPUTED")
    print("ML leakage audit: NOT YET COMPLETE")
    print("No files were modified.")
    print("=" * 70)

finally:
    con.close()

STEP 05.7A — INDEPENDENT DATA INTEGRITY AUDIT
Original Features: True
Injected Features: True
Ground Truth: True
Final ML Dataset: True

1. SOURCE DIMENSIONS & KEY INTEGRITY

original: 387,162 rows | 3,737 users | 85 columns
Duplicate keys: 0
Null keys: 0

injected: 69,634 rows | 670 users | 115 columns
Duplicate keys: 0
Null keys: 0

gt: 69,634 rows | 670 users | 13 columns
Duplicate keys: 0
Null keys: 0

final: 387,162 rows | 3,737 users | 116 columns
Duplicate keys: 0
Null keys: 0

2. SCHEMA COMPATIBILITY
Original columns: 85
Additional features: 30
Binary label: y
Final columns: 116

3. ORIGINAL RECORD PRESERVATION
Missing original keys: 0
Unexpected final keys: 0

4. NON-INJECTED USER VERIFICATION
Compared rows: 317,528
Compared columns: 85
Columns with mismatches: 0
Mismatch details: {}

5. INJECTED USER VERIFICATION
Compared rows: 69,634
Compared columns: 115
Columns with mismatches: 0
Mismatch details: {}

6. INDEPENDENT LABEL VERIFICATION
Labels checked: 387,162
Incorrect labe

## Step 05.7B — Independent Mathematical Feature Audit

### Objective
Independently recalculate all 30 new behavioral features
and compare them against the final saved ML dataset.

### Independent Inputs
- Daily activity counts and historical percentiles
- Original historical baseline CSV
- User-Day chronological ordering

### Audit Coverage
- All 3,737 users
- All 387,162 User-Day records
- All 30 additional behavioral features
- All four rolling windows: 5, 10, 20, 30 workdays

### Feature Groups
1. Cumulative Deviation
2. Combined Channel Intensity
3. Dominant Channel Switches
4. Channel Distribution Shift
5. Consecutive Elevated Days
6. Cross-Channel Deviation Days
7. Rolling Channel Totals

### Validation
- Recalculate feature values independently.
- Compare every computed value with the saved value.
- Compare expected and actual NaN positions.
- Detect invalid and infinite values.
- Report mismatch counts and examples.
- Use strict numerical tolerances.

### Safety
Read-only audit. No files are modified.

In [20]:
# ============================================================
# STEP 05.7B — INDEPENDENT MATHEMATICAL FEATURE AUDIT
# Corrected HTTP source-to-feature naming
# READ ONLY — NO FILE MODIFICATIONS
# ============================================================

from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view
import numpy as np
import pandas as pd
import duckdb

# ------------------------------------------------------------
# 1. Source paths
# ------------------------------------------------------------

ROOT = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)

FINAL_PATH = (
    ROOT / "Clear_Attack_Simulation"
    / "data" / "behavior_features_r62_ml_final.parquet"
)

BASELINE_CSV = (
    ROOT / "Data_Engineering" / "Data_engineering"
    / "data" / "features" / "user_baseline_r62_v1.csv"
)

WINDOWS = (5, 10, 20, 30)

# Original daily activity columns
COUNTS = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count"
]

P50_COLS = [
    "email_count_p50",
    "usb_copy_p50",
    "http_upload_p50"
]

P95_COLS = [
    "email_count_p95",
    "usb_copy_p95",
    "http_upload_p95"
]

HISTORY_COLS = [
    (
        "email_count_p95",
        "email_count_mean_active",
        "email_active_rate",
        "email_active_days"
    ),
    (
        "usb_copy_p95",
        "usb_copy_mean_active",
        "usb_active_rate",
        "usb_active_days"
    ),
    (
        "http_upload_p95",
        "http_upload_mean_active",
        "http_upload_active_rate",
        "http_upload_active_days"
    )
]

GROUPS = [
    "cumulative_deviation",
    "combined_channel_intensity",
    "dominant_channel_switches",
    "channel_distribution_shift",
    "consecutive_elevated_days",
    "cross_channel_deviation_days"
]

# IMPORTANT:
# Source column name != feature prefix for HTTP.
ROLLING_CHANNELS = [
    ("external_email_count", "external_email_count"),
    ("usb_copy_count", "usb_copy_count"),
    ("www_upload_count", "http_upload_count")
]

FEATURES_30 = (
    [
        f"{group}_{w}wd"
        for group in GROUPS
        for w in WINDOWS
    ]
    +
    [
        f"{prefix}_{w}wd"
        for w in (20, 30)
        for source, prefix in ROLLING_CHANNELS
    ]
)

assert len(FEATURES_30) == 30
assert len(set(FEATURES_30)) == 30

print("=" * 70)
print("STEP 05.7B — INDEPENDENT MATHEMATICAL AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 2. Load saved ML dataset independently
# ------------------------------------------------------------

assert FINAL_PATH.is_file()
assert BASELINE_CSV.is_file()

con = duckdb.connect(database=":memory:")

try:
    final = con.execute(
        "SELECT * FROM read_parquet(?)",
        [str(FINAL_PATH)]
    ).df()

finally:
    con.close()

history = pd.read_csv(BASELINE_CSV)

final["user"] = final["user"].astype(str)
final["day"] = pd.to_datetime(
    final["day"]
).dt.normalize()

history["user"] = history["user"].astype(str)

assert history["user"].is_unique

history = history.set_index("user")

final = (
    final.sort_values(
        ["user", "day"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

assert len(final) == 387_162
assert final["user"].nunique() == 3_737
assert len(final.columns) == 116

assert not final.duplicated(["user", "day"]).any()
assert final["is_weekend"].eq(0).all()

# Exact schema verification
missing_features = set(FEATURES_30) - set(final.columns)

assert not missing_features, (
    f"Missing features: {missing_features}"
)

assert "www_upload_count_20wd" not in FEATURES_30
assert "www_upload_count_30wd" not in FEATURES_30

assert "http_upload_count_20wd" in FEATURES_30
assert "http_upload_count_30wd" in FEATURES_30

assert set(final["user"]).issubset(history.index)

print("\nSOURCE VALIDATION")
print("Rows:", f"{len(final):,}")
print("Users:", f"{final['user'].nunique():,}")
print("Features to verify:", len(FEATURES_30))
print("HTTP naming: VERIFIED")

# ------------------------------------------------------------
# 3. Prepare independent source arrays
# ------------------------------------------------------------

all_counts = final[COUNTS].to_numpy(
    dtype=np.float64
)

all_p50 = final[P50_COLS].to_numpy(
    dtype=np.float64
)

all_p95 = final[P95_COLS].to_numpy(
    dtype=np.float64
)

assert np.isfinite(all_counts).all()
assert (all_counts >= 0).all()

expected = {
    feature: np.full(
        len(final),
        np.nan,
        dtype=np.float64
    )
    for feature in FEATURES_30
}

user_sizes = (
    final.groupby("user", sort=False)
    .size()
)

# ------------------------------------------------------------
# 4. Historical channel distributions
# ------------------------------------------------------------

def calculate_historical_shares(user):

    record = history.loc[user]

    weights = np.zeros(3, dtype=np.float64)

    for j, columns in enumerate(HISTORY_COLS):

        p95_name, mean_name, rate_name, days_name = columns

        p95 = float(record[p95_name])
        mean = float(record[mean_name])
        rate = float(record[rate_name])
        days = float(record[days_name])

        usable = (
            np.isfinite(p95)
            and p95 > 0
            and np.isfinite(mean)
            and mean > 0
            and np.isfinite(rate)
            and 0 < rate <= 1
            and np.isfinite(days)
            and days > 0
        )

        inactive = (
            days == 0
            and (rate == 0 or np.isnan(rate))
        )

        assert usable or inactive, (
            f"Invalid baseline: user={user}, channel={j}"
        )

        if usable:
            weights[j] = rate * mean / p95

    total = weights.sum()

    assert np.isfinite(total) and total > 0

    return weights / total

# ------------------------------------------------------------
# 5. Independently recalculate all 30 features
# ------------------------------------------------------------

start = 0
processed_users = 0

for user, n in user_sizes.items():

    stop = start + n

    counts = all_counts[start:stop]
    p50 = all_p50[start:stop]
    p95 = all_p95[start:stop]

    valid_p95 = (
        np.isfinite(p95)
        & (p95 > 0)
    )

    valid_deviation = (
        valid_p95
        & np.isfinite(p50)
        & (p50 > 0)
        & (p95 >= p50)
    )

    assert not (
        (counts > 0) & ~valid_deviation
    ).any(), f"Invalid activity baseline: {user}"

    assert valid_p95.any(axis=1).all()

    # --------------------------------------------------------
    # A. Daily cumulative deviation input
    # --------------------------------------------------------

    deviation = np.zeros_like(counts)

    np.divide(
        np.maximum(counts - p50, 0),
        np.maximum(p95 - p50, 1.0),
        out=deviation,
        where=valid_deviation
    )

    daily_deviation = deviation.sum(axis=1)

    # --------------------------------------------------------
    # B. Daily combined intensity input
    # --------------------------------------------------------

    normalized = np.zeros_like(counts)

    np.divide(
        counts,
        p95,
        out=normalized,
        where=valid_p95
    )

    daily_intensity = (
        normalized.sum(axis=1)
        / valid_p95.sum(axis=1)
    )

    # --------------------------------------------------------
    # C. Dominant channel input
    # --------------------------------------------------------

    eligible = np.where(
        valid_p95,
        normalized,
        -np.inf
    )

    maximum = eligible.max(axis=1)

    tied = np.isclose(
        eligible,
        maximum[:, None],
        rtol=1e-9,
        atol=1e-12
    )

    unique_positive = (
        (maximum > 0)
        & (tied.sum(axis=1) == 1)
    )

    dominant = np.full(n, -1, dtype=np.int8)

    dominant[unique_positive] = (
        eligible.argmax(axis=1)[unique_positive]
    )

    # --------------------------------------------------------
    # D. Elevated activity inputs
    # --------------------------------------------------------

    elevated = (
        valid_deviation
        & (counts > p50)
    )

    any_elevated = elevated.any(axis=1)

    cross_channel = (
        elevated.sum(axis=1) >= 2
    ).astype(np.int8)

    # --------------------------------------------------------
    # E. Historical distribution input
    # --------------------------------------------------------

    historical_shares = (
        calculate_historical_shares(user)
    )

    # --------------------------------------------------------
    # F. Calculate each complete rolling window
    # --------------------------------------------------------

    for w in WINDOWS:

        if n < w:
            continue

        target = slice(start + w - 1, stop)

        num_windows = n - w + 1

        # 1. Cumulative deviation
        deviation_windows = sliding_window_view(
            daily_deviation, w
        )

        expected[
            f"cumulative_deviation_{w}wd"
        ][target] = deviation_windows.sum(axis=1)

        # 2. Combined intensity
        intensity_windows = sliding_window_view(
            daily_intensity, w
        )

        expected[
            f"combined_channel_intensity_{w}wd"
        ][target] = intensity_windows.mean(axis=1)

        # 3. Dominant channel switches
        dominant_windows = sliding_window_view(
            dominant, w
        )

        previous = np.full(
            num_windows, -1, dtype=np.int8
        )

        switches = np.zeros(
            num_windows, dtype=np.float64
        )

        for position in range(w):

            current = dominant_windows[:, position]

            switches += (
                (current >= 0)
                & (previous >= 0)
                & (current != previous)
            )

            previous = np.where(
                current >= 0,
                current,
                previous
            )

        expected[
            f"dominant_channel_switches_{w}wd"
        ][target] = switches

        # 4. Channel distribution shift
        normalized_windows = sliding_window_view(
            normalized,
            window_shape=w,
            axis=0
        )

        channel_totals = (
            normalized_windows.sum(axis=2)
        )

        observed_total = (
            channel_totals.sum(axis=1)
        )

        valid_window = (
            np.isfinite(observed_total)
            & (observed_total > 0)
        )

        observed_shares = np.zeros_like(
            channel_totals
        )

        np.divide(
            channel_totals,
            observed_total[:, None],
            out=observed_shares,
            where=valid_window[:, None]
        )

        tvd = 0.5 * np.abs(
            observed_shares - historical_shares
        ).sum(axis=1)

        shifts = np.full(
            num_windows,
            np.nan,
            dtype=np.float64
        )

        shifts[valid_window] = np.clip(
            tvd[valid_window],
            0,
            1
        )

        expected[
            f"channel_distribution_shift_{w}wd"
        ][target] = shifts

        # 5. Consecutive elevated days
        elevated_windows = sliding_window_view(
            any_elevated, w
        )

        current_run = np.zeros(
            num_windows, dtype=np.int32
        )

        longest_run = np.zeros(
            num_windows, dtype=np.int32
        )

        for position in range(w):

            current_run = np.where(
                elevated_windows[:, position],
                current_run + 1,
                0
            )

            longest_run = np.maximum(
                longest_run,
                current_run
            )

        expected[
            f"consecutive_elevated_days_{w}wd"
        ][target] = longest_run

        # 6. Cross-channel deviation days
        cross_windows = sliding_window_view(
            cross_channel, w
        )

        expected[
            f"cross_channel_deviation_days_{w}wd"
        ][target] = cross_windows.sum(axis=1)

        # 7. Rolling channel totals
        if w in (20, 30):

            count_windows = sliding_window_view(
                counts,
                window_shape=w,
                axis=0
            )

            totals = count_windows.sum(axis=2)

            for j, (source, prefix) in enumerate(
                ROLLING_CHANNELS
            ):

                assert source == COUNTS[j]

                feature_name = f"{prefix}_{w}wd"

                assert feature_name in expected

                expected[feature_name][target] = (
                    totals[:, j]
                )

    start = stop
    processed_users += 1

    if processed_users % 750 == 0:
        print(
            f"Recalculated {processed_users:,} "
            f"of {len(user_sizes):,} users"
        )

assert start == len(final)

print("\nINDEPENDENT RECALCULATION COMPLETE")
print("Users processed:", f"{processed_users:,}")
print("Rows covered:", f"{start:,}")

# ------------------------------------------------------------
# 6. Compare every recalculated feature cell
# ------------------------------------------------------------

ABS_TOL = 1e-8
REL_TOL = 1e-9

total_mismatches = 0
total_nan_mismatches = 0
total_non_null = 0

audit_results = []
mismatch_examples = []

print("\n" + "=" * 70)
print("FEATURE-BY-FEATURE COMPARISON")
print("=" * 70)

for feature in FEATURES_30:

    actual = final[feature].to_numpy(
        dtype=np.float64
    )

    calculated = expected[feature]

    assert not np.isinf(actual).any(), (
        f"Infinite saved values: {feature}"
    )

    assert not np.isinf(calculated).any(), (
        f"Infinite calculated values: {feature}"
    )

    nan_mismatch = (
        np.isnan(actual)
        != np.isnan(calculated)
    )

    equal = np.isclose(
        actual,
        calculated,
        rtol=REL_TOL,
        atol=ABS_TOL,
        equal_nan=True
    )

    mismatch = ~equal

    mismatch_count = int(mismatch.sum())
    nan_count = int(nan_mismatch.sum())

    total_mismatches += mismatch_count
    total_nan_mismatches += nan_count

    total_non_null += int(
        np.isfinite(calculated).sum()
    )

    audit_results.append({
        "feature": feature,
        "mismatches": mismatch_count,
        "nan_mismatches": nan_count,
        "expected_nulls": int(
            np.isnan(calculated).sum()
        )
    })

    if mismatch_count:

        for pos in np.flatnonzero(mismatch)[:3]:

            mismatch_examples.append({
                "feature": feature,
                "user": final.at[pos, "user"],
                "day": str(final.at[pos, "day"].date()),
                "saved": actual[pos],
                "recalculated": calculated[pos]
            })

    status = (
        "PASS" if mismatch_count == 0 else "FAIL"
    )

    print(
        f"{status} | {feature}: "
        f"{mismatch_count:,} mismatches, "
        f"{nan_count:,} NaN mismatches"
    )

# ------------------------------------------------------------
# 7. Final audit report
# ------------------------------------------------------------

audit_df = pd.DataFrame(audit_results)

print("\n" + "=" * 70)
print("FINAL MATHEMATICAL AUDIT SUMMARY")
print("=" * 70)

print("Users audited:", f"{processed_users:,}")
print("Rows audited:", f"{len(final):,}")
print("Features audited:", len(FEATURES_30))

print(
    "Feature cells checked:",
    f"{len(final) * len(FEATURES_30):,}"
)

print(
    "Non-null values checked:",
    f"{total_non_null:,}"
)

print(
    "Feature mismatches:",
    f"{total_mismatches:,}"
)

print(
    "NaN position mismatches:",
    f"{total_nan_mismatches:,}"
)

if mismatch_examples:

    print("\nFIRST MISMATCH EXAMPLES")

    for example in mismatch_examples[:10]:
        print(example)

if total_mismatches == 0:

    print(
        "\nPASS — ALL 30 FEATURES "
        "INDEPENDENTLY VERIFIED"
    )

else:

    print(
        "\nFAIL — MATHEMATICAL DIFFERENCES FOUND"
    )

print("No files were modified.")
print("=" * 70)

assert total_mismatches == 0, (
    "Independent mathematical audit failed. "
    "Inspect mismatches before proceeding."
)

STEP 05.7B — INDEPENDENT MATHEMATICAL AUDIT

SOURCE VALIDATION
Rows: 387,162
Users: 3,737
Features to verify: 30
HTTP naming: VERIFIED
Recalculated 750 of 3,737 users
Recalculated 1,500 of 3,737 users
Recalculated 2,250 of 3,737 users
Recalculated 3,000 of 3,737 users

INDEPENDENT RECALCULATION COMPLETE
Users processed: 3,737
Rows covered: 387,162

FEATURE-BY-FEATURE COMPARISON
PASS | cumulative_deviation_5wd: 0 mismatches, 0 NaN mismatches
PASS | cumulative_deviation_10wd: 0 mismatches, 0 NaN mismatches
PASS | cumulative_deviation_20wd: 0 mismatches, 0 NaN mismatches
PASS | cumulative_deviation_30wd: 0 mismatches, 0 NaN mismatches
PASS | combined_channel_intensity_5wd: 0 mismatches, 0 NaN mismatches
PASS | combined_channel_intensity_10wd: 0 mismatches, 0 NaN mismatches
PASS | combined_channel_intensity_20wd: 0 mismatches, 0 NaN mismatches
PASS | combined_channel_intensity_30wd: 0 mismatches, 0 NaN mismatches
PASS | dominant_channel_switches_5wd: 0 mismatches, 0 NaN mismatches
PASS | d

## Step 05.7C — ML Quality & Leakage Risk Audit

### Objective
Audit the final ML dataset for statistical quality,
label leakage, temporal-window consistency, and evaluation risks.

### Checks
1. Inspect numeric missing values, infinity, and constant columns.
2. Compare missingness patterns between attack and non-attack days.
3. Detect explicit attack metadata or features identical to y.
4. Validate rolling-window warm-up behavior.
5. Identify non-attack days whose features contain earlier attack activity.
6. Inspect attack-user coverage and temporal distribution.
7. Record unresolved baseline look-ahead and data-splitting risks.

### Important Distinction
Passing this audit does not automatically prove that all temporal
leakage is absent. Historical baseline construction must also
be examined before final model evaluation.

### Safety
Read-only audit. No files are modified.

In [21]:
# ============================================================
# STEP 05.7C — ML QUALITY & LEAKAGE RISK AUDIT
# READ ONLY — NO FILE MODIFICATIONS
# ============================================================

from pathlib import Path
import duckdb
import pandas as pd
import re

ROOT = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)

FINAL_PATH = (
    ROOT / "Clear_Attack_Simulation"
    / "data" / "behavior_features_r62_ml_final.parquet"
)

assert FINAL_PATH.is_file()

def quoted(name):
    return '"' + name.replace('"', '""') + '"'

def sql_path(path):
    return path.as_posix().replace("'", "''")

con = duckdb.connect(database=":memory:")

hard_failures = []
review_flags = []

try:
    con.execute(f"""
        CREATE VIEW ml AS
        SELECT *
        FROM read_parquet('{sql_path(FINAL_PATH)}')
    """)

    schema = con.execute("""
        DESCRIBE SELECT * FROM ml
    """).fetchall()

    columns = [row[0] for row in schema]

    numeric_types = (
        "TINYINT", "SMALLINT", "INTEGER", "BIGINT",
        "HUGEINT", "UTINYINT", "USMALLINT",
        "UINTEGER", "UBIGINT", "FLOAT", "DOUBLE",
        "REAL", "DECIMAL"
    )

    numeric_columns = [
        name
        for name, dtype, *_ in schema
        if dtype.upper().startswith(numeric_types)
        and name not in ["y"]
    ]

    # --------------------------------------------------------
    # 1. Dataset and label verification
    # --------------------------------------------------------

    print("=" * 70)
    print("STEP 05.7C — ML QUALITY & LEAKAGE AUDIT")
    print("=" * 70)

    stats = con.execute("""
        SELECT
            COUNT(*),
            COUNT(DISTINCT user),
            COUNT(DISTINCT (user, day)),
            SUM(CASE WHEN y = 1 THEN 1 ELSE 0 END),
            SUM(CASE WHEN y = 0 THEN 1 ELSE 0 END),
            SUM(CASE
                WHEN y IS NULL OR y NOT IN (0, 1)
                THEN 1 ELSE 0 END)
        FROM ml
    """).fetchone()

    rows, users, keys, positives, negatives, bad_labels = stats

    print("\n1. DATASET INTEGRITY")
    print("Rows:", f"{rows:,}")
    print("Users:", f"{users:,}")
    print("Columns:", len(columns))
    print("Numeric feature columns:", len(numeric_columns))
    print("Duplicate keys:", rows - keys)
    print("Positive labels:", f"{positives:,}")
    print("Negative labels:", f"{negatives:,}")
    print("Invalid labels:", bad_labels)

    if (
        rows != 387_162
        or users != 3_737
        or len(columns) != 116
        or keys != rows
        or positives != 12_636
        or negatives != 374_526
        or bad_labels != 0
    ):
        hard_failures.append("Dataset dimensions or labels")

    # --------------------------------------------------------
    # 2. Full numeric-column quality profile
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("2. NUMERIC QUALITY PROFILE")
    print("=" * 70)

    profiles = []

    # Query in batches to control query size.
    for offset in range(0, len(numeric_columns), 12):

        batch = numeric_columns[offset:offset + 12]

        expressions = []

        for i, column in enumerate(batch):
            q = quoted(column)
            value = f"CAST({q} AS DOUBLE)"

            expressions.extend([
                f"COUNT(*) FILTER "
                f"(WHERE {q} IS NULL) AS null_{i}",

                f"COUNT(*) FILTER "
                f"(WHERE {q} IS NOT NULL "
                f"AND NOT isfinite({value})) AS invalid_{i}",

                f"MIN({value}) FILTER "
                f"(WHERE isfinite({value})) AS min_{i}",

                f"MAX({value}) FILTER "
                f"(WHERE isfinite({value})) AS max_{i}",

                f"COUNT(*) FILTER "
                f"(WHERE y = 1 AND {q} IS NULL) "
                f"AS attack_null_{i}"
            ])

        result = con.execute(
            "SELECT " + ", ".join(expressions) + " FROM ml"
        ).fetchone()

        for i, column in enumerate(batch):

            values = result[i * 5:(i + 1) * 5]

            nulls, invalids, minimum, maximum, attack_nulls = values

            profiles.append({
                "column": column,
                "nulls": int(nulls),
                "null_pct": 100 * nulls / rows,
                "attack_nulls": int(attack_nulls),
                "invalid_numeric": int(invalids),
                "min": minimum,
                "max": maximum,
                "constant": (
                    minimum is not None
                    and maximum is not None
                    and minimum == maximum
                )
            })

    profile_df = pd.DataFrame(profiles)

    print("Numeric columns profiled:", len(profile_df))

    print(
        "Columns containing NULL:",
        int((profile_df["nulls"] > 0).sum())
    )

    print(
        "Columns containing NaN/Infinity:",
        int((profile_df["invalid_numeric"] > 0).sum())
    )

    print(
        "Constant numeric columns:",
        int(profile_df["constant"].sum())
    )

    print("\nTOP 15 COLUMNS BY MISSINGNESS")

    print(
        profile_df.sort_values(
            "null_pct", ascending=False
        )[
            ["column", "nulls", "null_pct", "attack_nulls"]
        ].head(15).to_string(index=False)
    )

    constant_columns = profile_df.loc[
        profile_df["constant"], "column"
    ].tolist()

    print("\nCONSTANT COLUMNS")
    print(constant_columns)

    if profile_df["invalid_numeric"].sum() > 0:
        hard_failures.append("NaN/Infinity stored as numeric values")

    # Constant columns are review items, not automatic failures.
    if constant_columns:
        review_flags.append("Constant numeric columns")

    # --------------------------------------------------------
    # 3. Explicit metadata and label-copy screening
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("3. TARGET LEAKAGE SCREEN")
    print("=" * 70)

    suspicious_pattern = re.compile(
        r"(^|_)(attack|scenario|stealth|injected|"
        r"ground_truth|malicious|label|target)($|_)",
        re.IGNORECASE
    )

    suspicious_names = [
        column
        for column in columns
        if column != "y"
        and suspicious_pattern.search(column)
    ]

    print("Suspicious metadata columns:", suspicious_names)

    if suspicious_names:
        hard_failures.append("Explicit attack metadata columns")

    copied_label_columns = []
    inverted_label_columns = []

    for offset in range(0, len(numeric_columns), 20):

        batch = numeric_columns[offset:offset + 20]
        expressions = []

        for i, column in enumerate(batch):

            q = quoted(column)

            expressions.append(f"""
                COUNT(*) FILTER (
                    WHERE {q} IS DISTINCT FROM
                          CAST(y AS DOUBLE)
                ) AS same_{i}
            """)

            expressions.append(f"""
                COUNT(*) FILTER (
                    WHERE {q} IS DISTINCT FROM
                          CAST(1 - y AS DOUBLE)
                ) AS inverse_{i}
            """)

        result = con.execute(
            "SELECT " + ", ".join(expressions) + " FROM ml"
        ).fetchone()

        for i, column in enumerate(batch):

            if result[i * 2] == 0:
                copied_label_columns.append(column)

            if result[i * 2 + 1] == 0:
                inverted_label_columns.append(column)

    print("Features identical to y:", copied_label_columns)
    print("Features identical to 1-y:", inverted_label_columns)

    if copied_label_columns or inverted_label_columns:
        hard_failures.append("Direct label-copy features")

    print(
        "\nNote: This checks direct label copies and metadata, "
        "not every possible statistical proxy for y."
    )

    # --------------------------------------------------------
    # 4. Rolling-window chronology and warm-up audit
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("4. ROLLING-WINDOW CONSISTENCY")
    print("=" * 70)

    window_features = []

    for column in columns:

        match = re.search(r"_(5|10|20|30)wd$", column)

        if match:
            window_features.append(
                (column, int(match.group(1)))
            )

    # Only the 30 features added by Attack Simulation
    new_prefixes = (
        "cumulative_deviation_",
        "combined_channel_intensity_",
        "dominant_channel_switches_",
        "channel_distribution_shift_",
        "consecutive_elevated_days_",
        "cross_channel_deviation_days_",
        "external_email_count_",
        "usb_copy_count_",
        "http_upload_count_"
    )

    new_features = [
        (column, window)
        for column, window in window_features
        if column.startswith(new_prefixes)
        and (
            not column.startswith(
                (
                    "external_email_count_",
                    "usb_copy_count_",
                    "http_upload_count_"
                )
            )
            or window in (20, 30)
        )
    ]

    assert len(new_features) == 30, (
        f"Expected 30 new window features; "
        f"found {len(new_features)}"
    )

    warmup_expressions = []

    for i, (column, window) in enumerate(new_features):

        q = quoted(column)

        warmup_expressions.append(f"""
            COUNT(*) FILTER (
                WHERE day_position < {window}
                  AND {q} IS NOT NULL
            ) AS early_{i}
        """)

        warmup_expressions.append(f"""
            COUNT(*) FILTER (
                WHERE day_position >= {window}
                  AND {q} IS NULL
            ) AS late_{i}
        """)

    warmup_sql = f"""
        WITH numbered AS (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY user
                    ORDER BY day
                ) AS day_position
            FROM ml
        )
        SELECT
            {", ".join(warmup_expressions)}
        FROM numbered
    """

    warmup_values = con.execute(warmup_sql).fetchone()

    early_violations = {}
    unexpected_late_nulls = {}
    shift_late_nulls = {}

    for i, (column, window) in enumerate(new_features):

        early = int(warmup_values[2 * i])
        late = int(warmup_values[2 * i + 1])

        if early:
            early_violations[column] = early

        if column.startswith("channel_distribution_shift_"):
            shift_late_nulls[column] = late
        elif late:
            unexpected_late_nulls[column] = late

    print("New rolling features:", len(new_features))
    print("Values before complete windows:", early_violations)
    print("Unexpected NULLs after complete windows:",
          unexpected_late_nulls)
    print("Distribution-shift NULLs after window completion:",
          shift_late_nulls)

    if early_violations or unexpected_late_nulls:
        hard_failures.append("Rolling-window consistency")

    # --------------------------------------------------------
    # 5. Attack signal carried into non-attack days
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("5. POST-ATTACK WINDOW CARRYOVER")
    print("=" * 70)

    carryover = con.execute("""
        WITH chronological AS (
            SELECT
                user,
                day,
                y,

                COALESCE(
                    SUM(y) OVER (
                        PARTITION BY user
                        ORDER BY day
                        ROWS BETWEEN
                            29 PRECEDING AND 1 PRECEDING
                    ),
                    0
                ) AS attacks_in_previous_29_days

            FROM ml
        )

        SELECT
            COUNT(*) FILTER (
                WHERE y = 0
                  AND attacks_in_previous_29_days > 0
            ) AS non_attack_days_with_prior_attack,

            COUNT(DISTINCT user) FILTER (
                WHERE y = 0
                  AND attacks_in_previous_29_days > 0
            ) AS affected_users,

            COUNT(*) FILTER (
                WHERE y = 0
                  AND attacks_in_previous_29_days = 0
            ) AS clean_history_negative_days

        FROM chronological
    """).fetchone()

    print(
        "y=0 days with earlier attacks in 30-day context:",
        f"{carryover[0]:,}"
    )

    print(
        "Users affected:",
        f"{carryover[1]:,}"
    )

    print(
        "y=0 days without attacks in previous 29 days:",
        f"{carryover[2]:,}"
    )

    if carryover[0] > 0:
        review_flags.append(
            "Negative-label rows with previous attack activity"
        )

    # --------------------------------------------------------
    # 6. User-level attack distribution
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("6. USER-LEVEL ATTACK DISTRIBUTION")
    print("=" * 70)

    user_stats = con.execute("""
        WITH per_user AS (
            SELECT
                user,
                COUNT(*) AS observed_days,
                SUM(y) AS attack_days
            FROM ml
            GROUP BY user
        )

        SELECT
            COUNT(*) AS users,

            COUNT(*) FILTER (
                WHERE attack_days > 0
            ) AS positive_users,

            COUNT(*) FILTER (
                WHERE attack_days = 0
            ) AS zero_attack_users,

            COUNT(*) FILTER (
                WHERE observed_days < 30
            ) AS short_history_users,

            MIN(attack_days) AS min_attack_days,
            MAX(attack_days) AS max_attack_days

        FROM per_user
    """).fetchone()

    print("Total users:", user_stats[0])
    print("Users with attack days:", user_stats[1])
    print("Users with no attack days:", user_stats[2])
    print("Users with <30 rows:", user_stats[3])
    print("Maximum attack days for one user:", user_stats[5])

    if user_stats[1] != 670:
        hard_failures.append("Unexpected number of attacked users")

    # --------------------------------------------------------
    # 7. Temporal distribution of labels
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("7. TEMPORAL LABEL DISTRIBUTION")
    print("=" * 70)

    monthly = con.execute("""
        SELECT
            CAST(DATE_TRUNC('month', day) AS DATE)
                AS month,
            COUNT(*) AS records,
            SUM(y) AS attack_days,
            ROUND(100.0 * AVG(y), 3)
                AS attack_percentage
        FROM ml
        GROUP BY 1
        ORDER BY 1
    """).df()

    print(monthly.to_string(index=False))

    # --------------------------------------------------------
    # 8. Important unresolved methodological risks
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("8. METHODOLOGICAL LEAKAGE REVIEW")
    print("=" * 70)

    print(
        "User/day identifiers must be excluded from "
        "the numeric model input X."
    )

    print(
        "Train/test split must account for repeated "
        "observations from the same users."
    )

    print(
        "Historical baselines must be reviewed for "
        "future-data look-ahead."
    )

    print(
        "Rolling features can remain elevated after "
        "a synthetic attack day."
    )

    review_flags.append(
        "Historical baseline temporal provenance not yet verified"
    )

    review_flags.append(
        "Train/test split design not yet verified"
    )

    # --------------------------------------------------------
    # 9. Final diagnostic report
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("FINAL 05.7C AUDIT SUMMARY")
    print("=" * 70)

    print("\nHard failures:", len(hard_failures))
    for issue in hard_failures:
        print("FAIL:", issue)

    print("\nReview flags:", len(review_flags))
    for issue in review_flags:
        print("REVIEW:", issue)

    if hard_failures:
        print("\nFAIL — DATA QUALITY ISSUES FOUND")
    else:
        print("\nPASS — AUTOMATED QUALITY CHECKS")
        print(
            "NOTE — Temporal baseline and evaluation "
            "risks still require review."
        )

    print("\nNo files were modified.")
    print("=" * 70)

finally:
    con.close()

STEP 05.7C — ML QUALITY & LEAKAGE AUDIT

1. DATASET INTEGRITY
Rows: 387,162
Users: 3,737
Columns: 116
Numeric feature columns: 111
Duplicate keys: 0
Positive labels: 12,636
Negative labels: 374,526
Invalid labels: 0

2. NUMERIC QUALITY PROFILE
Numeric columns profiled: 111
Columns containing NULL: 40
Columns containing NaN/Infinity: 0
Constant numeric columns: 2

TOP 15 COLUMNS BY MISSINGNESS
                         column  nulls  null_pct  attack_nulls
                http_upload_mad 328257 84.785439          4410
                http_upload_p50 328257 84.785439          4410
       http_upload_to_p95_ratio 328257 84.785439          4410
                http_upload_p95 328257 84.785439          4410
                http_upload_p90 328257 84.785439          4410
                   usb_copy_p95 312211 80.640920          2035
                   usb_copy_p90 312211 80.640920          2035
                   usb_copy_p50 312211 80.640920          2035
                   usb_copy_mad 31221


### Dataset Inventory Update — Final Parquet Files

**Objective:** Update `dataset_inventory_attack_simulation_r62.csv` to reflect the final directory structure of `Clear_Attack_Simulation`.

**Actions:**
1. Correct the paths of the four existing Parquet datasets.
2. Add the two additional datasets from `data/final`.
3. Retrieve the actual row and column counts directly from Parquet metadata.
4. Validate that all six files exist before updating the inventory.
5. Save the updated CSV to `documentation/dictionary/`.

**Expected Output:**
- An updated inventory containing six datasets.
- Correct file paths and verified dataset dimensions.
- No modifications to the original Parquet files.

**Scope:** Documentation update only. No attack simulation, feature recalculation, or data modification.


In [3]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

root = Path(r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Clear_Attack_Simulation")

inventory_path = root / "documentation" / "dataset_inventory_attack_simulation_r62.csv"

# Actual Parquet locations
datasets = {
    "behavior_features_injected_final": "data/stages_output/behavior_features_injected_final.parquet",
    "ground_truth_users": "data/ground_truth/ground_truth_users.parquet",
    "ground_truth_channel_days": "data/ground_truth/ground_truth_channel_days.parquet",
    "ground_truth_user_days": "data/ground_truth/ground_truth_user_days.parquet",
    "behavior_features_r62_ml_final": "data/final/behavior_features_r62_ml_final.parquet",
    "user_day_injected_r62": "data/final/user_day_injected_r62.parquet",
}

# Validate every source before modifying the inventory
records = []

for name, relative_path in datasets.items():
    file_path = root / relative_path

    if not file_path.is_file():
        raise FileNotFoundError(f"Missing: {file_path}")

    metadata = pq.ParquetFile(file_path).metadata

    records.append({
        "dataset": name,
        "rows": metadata.num_rows,
        "columns": metadata.num_columns,
        "file_path": str(file_path)
    })

# Save updated inventory
if not inventory_path.is_file():
    raise FileNotFoundError(f"Inventory missing: {inventory_path}")

inventory = pd.DataFrame(records)

inventory.to_csv(
    inventory_path,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("PASS — Dataset inventory updated successfully")
print(f"Datasets documented: {len(inventory)}")
print(f"Saved to: {inventory_path}")

display(inventory)

PASS — Dataset inventory updated successfully
Datasets documented: 6
Saved to: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Clear_Attack_Simulation\documentation\dataset_inventory_attack_simulation_r62.csv


,dataset,rows,columns,file_path
0,behavior_features_injected_final,69634,115,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
1,ground_truth_users,670,12,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
2,ground_truth_channel_days,14375,11,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
3,ground_truth_user_days,69634,13,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
4,behavior_features_r62_ml_final,387162,116,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
5,user_day_injected_r62,1394010,34,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...



### M5 — File Manifest Update & SHA-256 Integrity Verification

**Objective:** Update the Stage 3.5 File Manifest to reflect the final `Clear_Attack_Simulation` directory structure and verify file integrity after project reorganization.

**Tasks:**
1. Preserve the original Stage 3.5 file paths, sizes, and SHA-256 fingerprints as historical evidence.
2. Locate the nine originally audited files in the organized project.
3. Calculate their current file sizes and SHA-256 hashes.
4. Compare current fingerprints against the approved Stage 3.5 fingerprints.
5. Include the two additional Parquet files from `data/final`, clearly marking them as outside the original Stage 3.5 audit scope.
6. Save the updated manifest in `documentation/`.

**Expected Output:**
- Updated manifest with historical and current file information.
- PASS/FAIL integrity status for the nine original files.
- Separate identification of newly added datasets.
- No modification to source datasets or the original QA results.

**Scope:** File organization and integrity verification only. This step does not repeat the independent mathematical or feature-level QA.


In [4]:

from pathlib import Path
import pandas as pd
import hashlib
import shutil

# Project paths
project_root = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration"
)

clear_root = project_root / "Clear_Attack_Simulation"
manifest_path = (
    clear_root
    / "validation_reports"
    / "Stage_3_5_QA_20261026T132721Z_9ee5d80e"
    / "MUBSIR_Stage3_5_File_Manifest.csv"
)
backup_path = manifest_path.with_name("MUBSIR_Stage3_5_File_Manifest_ORIGINAL.csv")

# Original audited files and their expected organized locations
file_locations = {
    "Original User-Day": project_root / "Data_Engineering/Data_engineering/data/processed/user_day_r62_v1.parquet",
    "Original Baseline": project_root / "Data_Engineering/Data_engineering/data/features/user_baseline_r62_v1.csv",
    "Approved Scenario Catalog": clear_root / "data/stages_output/approved_scenario_catalog_r62.parquet",
    "Validated Schedule": clear_root / "validation_inputs/validated_schedule_339.parquet",
    "GT Users": clear_root / "data/ground_truth/ground_truth_users.parquet",
    "GT Channel-Days": clear_root / "data/ground_truth/ground_truth_channel_days.parquet",
    "GT User-Days": clear_root / "data/ground_truth/ground_truth_user_days.parquet",
    "Final Features": clear_root / "data/stages_output/behavior_features_injected_final.parquet",
    "Historical Baseline Parquet": project_root / "Data_Engineering/Data_engineering/data/features/user_baseline_r62_v1.parquet"
}

# Additional final datasets (not part of the historical QA)
new_datasets = {
    "ML Final Features": clear_root / "data/final/behavior_features_r62_ml_final.parquet",
    "Injected User-Day": clear_root / "data/final/user_day_injected_r62.parquet"
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Load original manifest
if not manifest_path.is_file():
    raise FileNotFoundError(manifest_path)

manifest = pd.read_csv(manifest_path)

required_columns = [
    "Dataset", "File", "Size_Bytes",
    "Initial_SHA256", "Final_SHA256",
    "Unchanged_Since_3_5_1"
]

if not set(required_columns).issubset(manifest.columns):
    raise ValueError("Unexpected manifest structure")

# Prevent using an already modified manifest as historical evidence
if "Current_SHA256" in manifest.columns:
    if not backup_path.is_file():
        raise RuntimeError("Historical backup is missing")
    original = pd.read_csv(backup_path)
else:
    original = manifest.copy()

expected_names = set(file_locations)
if len(original) != 9 or set(original["Dataset"]) != expected_names:
    raise ValueError("Historical manifest does not match the expected nine files")

records = []

# Verify original audited files
for _, row in original.iterrows():
    name = row["Dataset"]
    path = file_locations[name]

    record = row.to_dict()
    record["Current_Path"] = str(path)
    record["Current_Size_Bytes"] = pd.NA
    record["Current_SHA256"] = pd.NA

    if not path.is_file():
        record["Integrity_Status"] = "MISSING"
    else:
        current_hash = sha256_file(path)
        current_size = path.stat().st_size

        record["Current_Size_Bytes"] = current_size
        record["Current_SHA256"] = current_hash

        expected_hash = row["Final_SHA256"]
        if (
            isinstance(expected_hash, str)
            and current_hash.lower() == expected_hash.lower()
            and current_size == int(row["Size_Bytes"])
        ):
            record["Integrity_Status"] = "PASS"
        else:
            record["Integrity_Status"] = "FAIL"

    records.append(record)

# Document additional final datasets separately
for name, path in new_datasets.items():
    record = {col: pd.NA for col in required_columns}
    record["Dataset"] = name
    record["File"] = pd.NA
    record["Current_Path"] = str(path)

    if path.is_file():
        record["Current_Size_Bytes"] = path.stat().st_size
        record["Current_SHA256"] = sha256_file(path)
        record["Integrity_Status"] = "NEW_NOT_IN_STAGE3_5"
    else:
        record["Current_Size_Bytes"] = pd.NA
        record["Current_SHA256"] = pd.NA
        record["Integrity_Status"] = "MISSING"

    records.append(record)

updated = pd.DataFrame(records)

# Save only after checking the original audit scope
historical_status = updated.iloc[:9]["Integrity_Status"]
missing = (updated["Integrity_Status"] == "MISSING").sum()
failed = (historical_status == "FAIL").sum()

if missing or failed:
    print("WARNING — Missing files or integrity differences detected.")
    print("Manifest NOT overwritten. Review the results first.")
else:
    # Keep the historical manifest for audit traceability
    if not backup_path.exists():
        shutil.copy2(manifest_path, backup_path)

    updated.to_csv(
        manifest_path,
        index=False,
        encoding="utf-8-sig"
    )
    print("PASS — File Manifest updated successfully")
    print(f"Saved to: {manifest_path}")
    print(f"Historical backup: {backup_path}")

print(f"Original files verified: {(historical_status == 'PASS').sum()}/9")
print(f"New datasets documented: {(updated['Integrity_Status'] == 'NEW_NOT_IN_STAGE3_5').sum()}/2")

display(updated[[
    "Dataset",
    "Current_Size_Bytes",
    "Integrity_Status",
    "Current_Path"
]])


PASS — File Manifest updated successfully
Saved to: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Clear_Attack_Simulation\validation_reports\Stage_3_5_QA_20261026T132721Z_9ee5d80e\MUBSIR_Stage3_5_File_Manifest.csv
Historical backup: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Clear_Attack_Simulation\validation_reports\Stage_3_5_QA_20261026T132721Z_9ee5d80e\MUBSIR_Stage3_5_File_Manifest_ORIGINAL.csv
Original files verified: 9/9
New datasets documented: 2/2


,Dataset,Current_Size_Bytes,Integrity_Status,Current_Path
0,Original User-Day,24648454,PASS,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
1,Original Baseline,928190,PASS,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
2,Approved Scenario Catalog,25236,PASS,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
3,Validated Schedule,67204,PASS,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
4,GT Users,22176,PASS,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
5,GT Channel-Days,71935,PASS,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
6,GT User-Days,92480,PASS,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
7,Final Features,7376590,PASS,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
8,Historical Baseline Parquet,221908,PASS,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...
9,ML Final Features,17559307,NEW_NOT_IN_STAGE3_5,C:\Users\hp\Documents\FYP_Data_Exfiltration_Or...



### M6 — Final Project README Update

**Objective:** Update the main `README.md` to reflect the final approved structure of `Clear_Attack_Simulation`.

**Tasks:**
1. Document the actual project directory structure.
2. Describe the four main simulation notebooks and the additional reconstruction notebook.
3. Identify the six Parquet datasets and their correct locations.
4. Explain the distinction between final ML datasets, Ground Truth, and intermediate stage outputs.
5. Document the validated attack schedule and its purpose.
6. Include the verified Stage 3.5 QA results (19/19 PASS).
7. Reference the updated Dataset Inventory, File Manifest, and data dictionaries.
8. Preserve a backup of the previous README before updating it.

**Expected Output:**
- Updated root `README.md`.
- Correct dataset paths and actual dimensions.
- Clear documentation of simulation outputs and validation evidence.
- No changes to source datasets, notebooks, or QA results.

**Scope:** Documentation synchronization only.


In [5]:

from pathlib import Path
import json
import shutil
import pandas as pd
import pyarrow.parquet as pq

# ============================================================
# M6 — FINAL README UPDATE
# ============================================================

root = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\Clear_Attack_Simulation"
)

readme_path = root / "README.md"
backup_path = root / "documentation" / "README_BEFORE_M6.md"

inventory_path = (
    root / "documentation"
    / "dataset_inventory_attack_simulation_r62.csv"
)

qa_dir = (
    root / "validation_reports"
    / "Stage_3_5_QA_20261026T132721Z_9ee5d80e"
)

summary_path = qa_dir / "MUBSIR_Stage3_5_QA_Summary.json"
manifest_path = qa_dir / "MUBSIR_Stage3_5_File_Manifest.csv"

# ------------------------------------------------------------
# 1. Validate required files
# ------------------------------------------------------------

required = [
    readme_path,
    inventory_path,
    summary_path,
    manifest_path
]

for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)

# ------------------------------------------------------------
# 2. Load approved results
# ------------------------------------------------------------

with open(summary_path, encoding="utf-8") as f:
    qa = json.load(f)

if qa["overall_status"] != "PASS":
    raise ValueError("Stage 3.5 QA status is not PASS")

inventory = pd.read_csv(inventory_path, sep=";")
manifest = pd.read_csv(manifest_path)

expected_datasets = {
    "behavior_features_injected_final":
        "data/stages_output/behavior_features_injected_final.parquet",
    "ground_truth_users":
        "data/ground_truth/ground_truth_users.parquet",
    "ground_truth_channel_days":
        "data/ground_truth/ground_truth_channel_days.parquet",
    "ground_truth_user_days":
        "data/ground_truth/ground_truth_user_days.parquet",
    "behavior_features_r62_ml_final":
        "data/final/behavior_features_r62_ml_final.parquet",
    "user_day_injected_r62":
        "data/final/user_day_injected_r62.parquet"
}

if set(inventory["dataset"]) != set(expected_datasets):
    raise ValueError("Inventory dataset names do not match")

# ------------------------------------------------------------
# 3. Verify actual Parquet metadata
# ------------------------------------------------------------

dataset_lines = []

for name, relative_path in expected_datasets.items():
    path = root / relative_path

    if not path.is_file():
        raise FileNotFoundError(path)

    metadata = pq.ParquetFile(path).metadata

    recorded = inventory.loc[
        inventory["dataset"] == name
    ].iloc[0]

    if (
        int(recorded["rows"]) != metadata.num_rows
        or int(recorded["columns"]) != metadata.num_columns
    ):
        raise ValueError(f"Inventory mismatch: {name}")

    dataset_lines.append(
        f"| `{name}` | `{relative_path}` | "
        f"{metadata.num_rows:,} | {metadata.num_columns} |"
    )

# ------------------------------------------------------------
# 4. Verify notebook and documentation locations
# ------------------------------------------------------------

notebooks = [
    "01_scenario_rules_and_feasibility.ipynb",
    "02_user_eligibility_and_assignment.ipynb",
    "03_attack_scheduling_and_injection.ipynb",
    "04_independent_end_to_end_validation.ipynb",
    "Reconstruct Injected User-Day.ipynb"
]

for name in notebooks:
    if not (root / "notebooks" / name).is_file():
        raise FileNotFoundError(name)

schedule_path = (
    root / "validation_inputs" / "validated_schedule_339.parquet"
)

if not schedule_path.is_file():
    raise FileNotFoundError(schedule_path)

# ------------------------------------------------------------
# 5. Check updated manifest
# ------------------------------------------------------------

if "Integrity_Status" not in manifest.columns:
    raise ValueError("Updated manifest is missing Integrity_Status")

original_checks = manifest[
    manifest["Integrity_Status"] != "NEW_NOT_IN_STAGE3_5"
]

if (
    len(original_checks) != 9
    or not original_checks["Integrity_Status"].eq("PASS").all()
):
    raise ValueError("Historical file integrity is not 9/9 PASS")

# ------------------------------------------------------------
# 6. Generate final README
# ------------------------------------------------------------

dataset_table = "\n".join(dataset_lines)

readme = f"""# MUBSIR — Clear Attack Simulation

**Dataset:** CERT r6.2  
**Status:** Approved Simulation Outputs  
**Injection Formula:** {qa['formula']}  
**Independent QA:** {qa['mandatory_checks_passed']}/{qa['mandatory_checks']} PASS

## 1. Project Overview

MUBSIR investigates low-and-slow data exfiltration across
Email, USB, and HTTP channels.

The Attack Simulation component generates synthetic
event-count additions using validated attack scenarios,
user-specific schedules, and Formula B.

The simulation operates on aggregated behavioral data.
It does not create raw synthetic event IDs or payload bytes.

This folder contains the organized simulation outputs,
supporting documentation, and independent validation evidence.

## 2. Project Structure

```text
Clear_Attack_Simulation/
|
|-- data/
|   |-- final/
|   |-- ground_truth/
|   `-- stages_output/
|
|-- notebooks/
|   |-- 01_scenario_rules_and_feasibility.ipynb
|   |-- 02_user_eligibility_and_assignment.ipynb
|   |-- 03_attack_scheduling_and_injection.ipynb
|   |-- 04_independent_end_to_end_validation.ipynb
|   `-- Reconstruct Injected User-Day.ipynb
|
|-- validation_inputs/
|   `-- validated_schedule_339.parquet
|
|-- validation_reports/
|   `-- Stage_3_5_QA_20261026T132721Z_9ee5d80e/
|
|-- documentation/
|   |-- dataset_inventory_attack_simulation_r62.csv
|   |-- dictionary/
|   |-- Explanations/
|   |-- ORGANIZATION_VALIDATION.txt
|   `-- README.md
|
`-- README.md
```

## 3. Simulation Workflow

### Stage 1 — Scenario Design

Notebook: `01_scenario_rules_and_feasibility.ipynb`

Defines and validates attack scenarios, including
duration, Off days, channel combinations, and switching rules.

Approved scenarios: 1,143.

### Stage 2 — User Eligibility and Assignment

Notebook: `02_user_eligibility_and_assignment.ipynb`

Evaluates users and assigns preliminary scenarios.

Preliminary assignments: 995 users.

These are not all confirmed injected attacks.

### Stage 3 — Attack Scheduling and Injection

Notebook: `03_attack_scheduling_and_injection.ipynb`

Constructs validated schedules for 670 users,
evaluates Formula A, B, and C, and applies Formula B.

The simulation produces updated behavioral features
and three Ground Truth datasets.

### Stage 3.5 — Independent Validation

Notebook: `04_independent_end_to_end_validation.ipynb`

Independently verifies schedule consistency,
injection calculations, feature reconstruction,
Ground Truth integrity, and historical baseline preservation.

### Additional Reconstruction Notebook

Notebook: `Reconstruct Injected User-Day.ipynb`

An additional notebook in the organized package.
Its exact processing scope and outputs should be
interpreted from its own executed cells and results;
it is not part of the original four-notebook
Stage 1–3.5 sequence.

## 4. Dataset Inventory

The following files exist in the organized package.
Dimensions are read directly from Parquet metadata.

| Dataset | Relative Path | Rows | Columns |
|---|---|---:|---:|
{dataset_table}

### Data Folder Roles

**data/stages_output**

Contains approved scenario artifacts and the
69,634-row simulated behavioral feature dataset.

**data/ground_truth**

Contains the three Ground Truth datasets:
user-level, channel-day-level, and user-day-level.

**data/final**

Contains the two additional final data products:
`behavior_features_r62_ml_final.parquet`
and `user_day_injected_r62.parquet`.

These files have different row counts from the
670-user simulation cohort. Their presence does not
establish that they are equivalent to the smaller
Stage 3.5-validated feature dataset.

## 5. Attack Injection Method

Selected formula: **Formula B — Natural Activity Addition**

```text
Injected Count = max(1, ceil(Natural Count * Multiplier))
Final Count = Natural Count + Injected Count
```

Stealth multipliers:

- High: 0.5
- Medium: 1.0
- Low: 1.5

Injection occurs only in a channel with natural
activity on the corresponding user-day.

The historical baseline and original byte-related
measurements are preserved.

## 6. Validated Schedule

File:

`validation_inputs/validated_schedule_339.parquet`

The schedule is a planning and validation input,
not a direct ML training dataset.

It records the validated attack timetable,
including Attack and Off days and planned channel behavior.

Scheduled Attack days: {qa['attack_days']:,}

Scheduled Off days: {qa['off_days']:,}

## 7. Ground Truth and ML Guidance

The original Stage 3 simulation outputs are:

- `data/stages_output/behavior_features_injected_final.parquet`
- `data/ground_truth/ground_truth_users.parquet`
- `data/ground_truth/ground_truth_channel_days.parquet`
- `data/ground_truth/ground_truth_user_days.parquet`

For the Stage 3 user-day detection task, join
`behavior_features_injected_final` with
`ground_truth_user_days` on `(user, day)`.

The binary label is `is_attack_day`.

Do not use scenario identifiers, stealth levels,
planned channels, injection amounts, or other
Ground Truth metadata as model predictors.

A value of `is_attack_day=0` means no synthetic
injection occurred on that day; it does not prove
all underlying CERT activity was benign.

The five original malicious CERT users were excluded
from the synthetic simulation cohort.

The additional files in `data/final` should be
examined separately before selecting an ML training dataset.

## 8. Independent QA Results

Source: Stage 3.5 QA Summary.

| Metric | Verified Result |
|---|---:|
| Simulated Users | {qa['simulated_users']:,} |
| Attack Days | {qa['attack_days']:,} |
| Off Days | {qa['off_days']:,} |
| Injected Channel-Days | {qa['injected_channel_days']:,} |
| Injected Event Counts | {qa['injected_event_counts']:,} |
| Original Columns | {qa['original_columns']} |
| Added Features | {qa['new_features']} |
| Final Columns | {qa['final_columns']} |
| Compared Values | {qa['total_compared_values']:,} |
| Unexplained Mismatches | {qa['unexplained_mismatches']} |
| Mandatory Checks Passed | {qa['mandatory_checks_passed']}/{qa['mandatory_checks']} |

These results certify the original Stage 3.5
audit scope. They do not automatically certify
the additional datasets under `data/final`.

## 9. File Integrity

The updated Stage 3.5 File Manifest records:

- Nine historical files verified against saved SHA-256 hashes.
- Two additional final datasets documented separately.

Manifest location:

`validation_reports/Stage_3_5_QA_20261026T132721Z_9ee5d80e/MUBSIR_Stage3_5_File_Manifest.csv`

The historical SHA-256 checks confirm file
identity, not independent feature-level QA for new datasets.

## 10. Documentation

- `documentation/dataset_inventory_attack_simulation_r62.csv`
  — dataset dimensions and locations.

- `documentation/dictionary/`
  — feature, schema, and Ground Truth dictionaries.

- `documentation/Explanations/`
  — formula selection, ML handoff, and file decisions.

- `documentation/ORGANIZATION_VALIDATION.txt`
  — historical project organization verification.

- `validation_reports/`
  — independent Stage 3.5 QA evidence.

## 11. Execution Warning

The notebooks preserve historical execution logic.
Some cells may reference earlier working directories.

Do not run all notebook cells without reviewing
and configuring their source and output paths.

The saved Parquet files and QA evidence are the
delivered results. Documentation changes must not
modify the original datasets.

## 12. Validation Scope

The original attack simulation and its Stage 3.5
validation are complete.

The two additional final Parquet datasets are
documented in the current inventory and manifest,
but require their own content-level checks before
any broader validation claim is made.
"""

# ------------------------------------------------------------
# 7. Save backup and update README
# ------------------------------------------------------------

backup_path.parent.mkdir(parents=True, exist_ok=True)

if not backup_path.exists():
    shutil.copy2(readme_path, backup_path)

readme_path.write_text(readme, encoding="utf-8")

print("PASS — Main README updated successfully")
print("README:", readme_path)
print("Backup:", backup_path)
print("Datasets documented:", len(dataset_lines))
print(
    "QA:",
    f"{qa['mandatory_checks_passed']}/{qa['mandatory_checks']} PASS"
)
print("Historical file integrity: 9/9 PASS")


PASS — Main README updated successfully
README: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Clear_Attack_Simulation\README.md
Backup: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Clear_Attack_Simulation\documentation\README_BEFORE_M6.md
Datasets documented: 6
QA: 19/19 PASS
Historical file integrity: 9/9 PASS



### M6 — Final README Documentation

**Objective:** Generate an accurate project README based on the verified `Clear_Attack_Simulation` folder structure and approved simulation results.

**Tasks:**
1. Verify the actual directory structure and notebook files.
2. Document the simulation workflow (Stages 1, 2, 3, and 3.5).
3. List the final datasets, Ground Truth files, and intermediate outputs.
4. Retrieve dataset dimensions directly from Parquet metadata.
5. Include the approved Formula B and Stage 3.5 validation results.
6. Document the roles of the validated schedule and QA reports.
7. Preserve the existing README before replacing it.

**Output:** `Clear_Attack_Simulation/README.md`

**Scope:** Documentation only. No Parquet files, notebooks, or QA evidence will be modified.


In [6]:

from pathlib import Path
from datetime import datetime
import json
import shutil
import pyarrow.parquet as pq

# ============================================================
# M6 — README GENERATION FROM VERIFIED PROJECT STRUCTURE
# ============================================================

root = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\Clear_Attack_Simulation"
)

readme_path = root / "README.md"
docs = root / "documentation"

qa_dir = (
    root / "validation_reports"
    / "Stage_3_5_QA_20261026T132721Z_9ee5d80e"
)

# ------------------------------------------------------------
# 1. Expected project files
# ------------------------------------------------------------

datasets = {
    "Final ML Features":
        "data/final/behavior_features_r62_ml_final.parquet",
    "Injected User-Day":
        "data/final/user_day_injected_r62.parquet",
    "GT Channel-Days":
        "data/ground_truth/ground_truth_channel_days.parquet",
    "GT User-Days":
        "data/ground_truth/ground_truth_user_days.parquet",
    "GT Users":
        "data/ground_truth/ground_truth_users.parquet",
    "Stage 3 Injected Features":
        "data/stages_output/behavior_features_injected_final.parquet",
    "Approved Scenario Catalog":
        "data/stages_output/approved_scenario_catalog_r62.parquet",
    "Validated Scenario Assignments":
        "data/stages_output/validated_scenario_assignments_r62.parquet",
    "Validated Schedule":
        "validation_inputs/validated_schedule_339.parquet",
}

notebooks = [
    "01_scenario_rules_and_feasibility.ipynb",
    "02_user_eligibility_and_assignment.ipynb",
    "03_attack_scheduling_and_injection.ipynb",
    "04_independent_end_to_end_validation.ipynb",
    "Reconstruct Injected User-Day.ipynb",
]

other_files = [
    "data/stages_output/approved_attack_scenarios_r62.csv",
    "data/stages_output/approved_switching_ranges.csv",
    "data/stages_output/stage2_assignment_validation_manifest_r62.json",
    "documentation/dataset_inventory_attack_simulation_r62.csv",
    "documentation/MUBSIR_Injection_Formula_Selection_AR.docx",
    "documentation/dictionary/feature_dictionary_attack_simulation_r62.csv",
    "documentation/dictionary/ground_truth_dictionary_attack_simulation_r62.csv",
    "documentation/dictionary/new_features_dictionary_30_r62.csv",
    "documentation/dictionary/schema_dictionary_attack_simulation_r62.csv",
]

qa_files = [
    "MUBSIR_Stage3_5_File_Manifest.csv",
    "MUBSIR_Stage3_5_Independent_QA_Report.html",
    "MUBSIR_Stage3_5_QA_Checks.csv",
    "MUBSIR_Stage3_5_QA_Summary.json",
]

required_paths = (
    [root / p for p in datasets.values()]
    + [root / "notebooks" / p for p in notebooks]
    + [root / p for p in other_files]
    + [qa_dir / p for p in qa_files]
)

missing = [str(p) for p in required_paths if not p.is_file()]

if missing:
    raise FileNotFoundError(
        "README not modified. Missing files:\n"
        + "\n".join(missing)
    )

# ------------------------------------------------------------
# 2. Read verified metadata
# ------------------------------------------------------------

dataset_rows = []

for name, relative_path in datasets.items():
    metadata = pq.ParquetFile(root / relative_path).metadata

    dataset_rows.append(
        f"| {name} | `{relative_path}` | "
        f"{metadata.num_rows:,} | {metadata.num_columns} |"
    )

dataset_table = "\n".join(dataset_rows)

summary_path = qa_dir / "MUBSIR_Stage3_5_QA_Summary.json"

with open(summary_path, encoding="utf-8") as f:
    qa = json.load(f)

if (
    qa["overall_status"] != "PASS"
    or qa["mandatory_checks"] != 19
    or qa["mandatory_checks_passed"] != 19
):
    raise ValueError("Unexpected Stage 3.5 QA status")

# ------------------------------------------------------------
# 3. Generate accurate README
# ------------------------------------------------------------

readme = f"""# MUBSIR — Clear Attack Simulation

**Dataset:** CERT r6.2  
**Component:** Multi-Channel Attack Simulation  
**Status:** Stage 3.5 Validated  
**Approved Injection Formula:** {qa['formula']}

## 1. Project Overview

MUBSIR investigates low-and-slow data exfiltration
across Email, USB, and HTTP channels.

The Attack Simulation component models synthetic
exfiltration behavior by increasing aggregated daily
event counts according to validated attack schedules.

The simulation does not create raw synthetic events,
individual event identifiers, or payload bytes.

This folder contains the organized project notebooks,
simulation datasets, Ground Truth, documentation,
and independent QA evidence.

## 2. Project Directory Structure

```text
Clear_Attack_Simulation/
├── data/
│   ├── final/
│   │   ├── behavior_features_r62_ml_final.parquet
│   │   └── user_day_injected_r62.parquet
│   ├── ground_truth/
│   │   ├── ground_truth_channel_days.parquet
│   │   ├── ground_truth_user_days.parquet
│   │   └── ground_truth_users.parquet
│   └── stages_output/
│       ├── approved_attack_scenarios_r62.csv
│       ├── approved_scenario_catalog_r62.parquet
│       ├── approved_switching_ranges.csv
│       ├── behavior_features_injected_final.parquet
│       ├── stage2_assignment_validation_manifest_r62.json
│       └── validated_scenario_assignments_r62.parquet
├── documentation/
│   ├── dictionary/
│   │   ├── feature_dictionary_attack_simulation_r62.csv
│   │   ├── ground_truth_dictionary_attack_simulation_r62.csv
│   │   ├── new_features_dictionary_30_r62.csv
│   │   └── schema_dictionary_attack_simulation_r62.csv
│   ├── dataset_inventory_attack_simulation_r62.csv
│   ├── MUBSIR_Injection_Formula_Selection_AR.docx
│   └── README_BEFORE_M6.md
├── notebooks/
│   ├── 01_scenario_rules_and_feasibility.ipynb
│   ├── 02_user_eligibility_and_assignment.ipynb
│   ├── 03_attack_scheduling_and_injection.ipynb
│   ├── 04_independent_end_to_end_validation.ipynb
│   └── Reconstruct Injected User-Day.ipynb
├── validation_inputs/
│   └── validated_schedule_339.parquet
├── validation_reports/
│   └── Stage_3_5_QA_20261026T132721Z_9ee5d80e/
│       ├── MUBSIR_Stage3_5_File_Manifest.csv
│       ├── MUBSIR_Stage3_5_Independent_QA_Report.html
│       ├── MUBSIR_Stage3_5_QA_Checks.csv
│       └── MUBSIR_Stage3_5_QA_Summary.json
└── README.md
```

## 3. Simulation Workflow

### Stage 1 — Scenario Design and Feasibility

Notebook: `notebooks/01_scenario_rules_and_feasibility.ipynb`

Defines attack scenarios, duration constraints,
Off days, channel combinations, and switching rules.

Approved attack scenarios: **1,143**.

### Stage 2 — User Eligibility and Assignment

Notebook: `notebooks/02_user_eligibility_and_assignment.ipynb`

Evaluates candidate users and assigns scenarios.

Preliminary assigned users: **995**.

The preliminary assignments must not be confused
with the final 670 users who received validated schedules.

### Stage 3 — Attack Scheduling and Injection

Notebook: `notebooks/03_attack_scheduling_and_injection.ipynb`

Constructs validated attack schedules, evaluates
injection quantity formulas A, B, and C,
and applies the selected Formula B.

Final simulated users: **{qa['simulated_users']:,}**.

Produces injected behavioral features and
three Ground Truth datasets.

### Stage 3.5 — Independent QA

Notebook: `notebooks/04_independent_end_to_end_validation.ipynb`

Independently validates injection calculations,
schedule consistency, reconstructed features,
Ground Truth, historical baseline preservation,
and saved file integrity.

### Additional User-Day Reconstruction

Notebook: `notebooks/Reconstruct Injected User-Day.ipynb`

Additional reconstruction work associated with
the organized project.

This notebook is documented separately from the
original Stage 1–3.5 execution sequence.
Its exact outputs should be interpreted from its
own code and execution results.

## 4. Dataset Inventory

All dimensions below were read directly
from the available Parquet metadata.

| Dataset | Relative Path | Rows | Columns |
|---|---|---:|---:|
{dataset_table}

### Dataset Organization

**data/stages_output/** contains approved scenario
artifacts, preliminary user assignments, and
the Stage 3 injected behavioral features.

**data/ground_truth/** contains user-level,
user-day-level, and channel-day-level
synthetic attack Ground Truth.

**data/final/** contains two additional datasets:
the ML final feature file and injected User-Day file.

These datasets have a broader row scope than the
670-user Stage 3 simulation output. Their file
dimensions alone do not establish equivalent
validation coverage.

## 5. Injection Method

Selected formula: **Formula B — Natural Activity Addition**

```text
Injected Count = max(1, ceil(Natural Count × Multiplier))
Final Count = Natural Count + Injected Count
```

Stealth multipliers:

- High Stealth: 0.5
- Medium Stealth: 1.0
- Low Stealth: 1.5

Synthetic event counts are added only to channels
with existing natural activity on the same user-day.

Original historical baseline values and
byte-related measurements remain preserved.

## 6. Validated Attack Schedule

File: `validation_inputs/validated_schedule_339.parquet`

The schedule is an approved planning and
validation input.

It records scheduled attack and Off days
and planned channel activity.

- Attack days: **{qa['attack_days']:,}**
- Off days: **{qa['off_days']:,}**

This file is not intended as an ML predictor.

## 7. Ground Truth and ML Guidance

Stage 3 feature dataset:

`data/stages_output/behavior_features_injected_final.parquet`

Daily labels:

`data/ground_truth/ground_truth_user_days.parquet`

For the original Stage 3 daily detection task,
join these files on `(user, day)`.

The binary target is `is_attack_day`.

Do not use Ground Truth or scenario metadata
as training predictors.

An `is_attack_day=0` label indicates no
synthetic injection on that user-day.
It does not establish that all underlying
source activity was benign.

The five original malicious CERT users
were excluded from the synthetic attack cohort.

The additional datasets in `data/final/`
should be assessed independently before
selecting ML training inputs.

## 8. Independent Validation Results

Source: `MUBSIR_Stage3_5_QA_Summary.json`

| Metric | Verified Result |
|---|---:|
| Simulated Users | {qa['simulated_users']:,} |
| Attack Days | {qa['attack_days']:,} |
| Off Days | {qa['off_days']:,} |
| Injected Channel-Days | {qa['injected_channel_days']:,} |
| Injected Event Counts | {qa['injected_event_counts']:,} |
| Original Feature Columns | {qa['original_columns']} |
| Newly Engineered Features | {qa['new_features']} |
| Final Feature Columns | {qa['final_columns']} |
| Compared Values | {qa['total_compared_values']:,} |
| Unexplained Mismatches | {qa['unexplained_mismatches']} |
| Mandatory Checks Passed | {qa['mandatory_checks_passed']}/{qa['mandatory_checks']} |

The original Stage 3.5 audit completed
**19/19 mandatory checks successfully**.

These results apply to the original audited
simulation outputs. They do not automatically
certify the additional final datasets.

## 9. File Integrity and QA Reports

The QA report directory contains:

- `MUBSIR_Stage3_5_File_Manifest.csv`
  — file paths, sizes, and SHA-256 integrity records.

- `MUBSIR_Stage3_5_Independent_QA_Report.html`
  — detailed visual QA report.

- `MUBSIR_Stage3_5_QA_Checks.csv`
  — individual mandatory quality checks.

- `MUBSIR_Stage3_5_QA_Summary.json`
  — compact machine-readable QA summary.

The updated File Manifest records nine historical
file integrity matches and two additional final
datasets marked outside the original QA scope.

## 10. Documentation

**documentation/dictionary/** contains the
feature, new-feature, Ground Truth, and schema dictionaries.

**documentation/dataset_inventory_attack_simulation_r62.csv**
records the six main Parquet datasets,
their verified dimensions, and current paths.

**documentation/MUBSIR_Injection_Formula_Selection_AR.docx**
documents the formula evaluation and selection.

**documentation/README_BEFORE_M6.md**
preserves an earlier README version.

## 11. Execution Warning

The notebooks retain historical execution logic.
Some original paths may reference the previous
Attack_Simulation working directory.

Do not run all cells without checking
source paths and output destinations.

The existing exported datasets and QA reports
must be preserved unless an intentional
reconstruction or validation run is required.

## 12. Validation Scope

The Stage 3 synthetic simulation and its
independent Stage 3.5 validation are complete.

The additional datasets inside `data/final/`
are documented as part of the organized project,
but their broader contents are not automatically
covered by the original Stage 3.5 QA results.
"""

# ------------------------------------------------------------
# 4. Preserve current README, then save
# ------------------------------------------------------------

docs.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = docs / f"README_BACKUP_{timestamp}.md"

if readme_path.exists():
    shutil.copy2(readme_path, backup_path)

readme_path.write_text(readme, encoding="utf-8")

print("PASS — README updated successfully")
print("README:", readme_path)
print("Backup:", backup_path)
print("Datasets documented:", len(datasets))
print("QA:", f"{qa['mandatory_checks_passed']}/{qa['mandatory_checks']} PASS")


PASS — README updated successfully
README: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Clear_Attack_Simulation\README.md
Backup: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Clear_Attack_Simulation\documentation\README_BACKUP_20261028_162652.md
Datasets documented: 9
QA: 19/19 PASS


In [7]:
from pathlib import Path

root = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\Clear_Attack_Simulation"
)

readme_path = root / "README.md"

content = readme_path.read_text(encoding="utf-8")

# Remove obsolete backup references
lines_to_remove = [
    "│   └── README_BEFORE_M6.md",
    "**documentation/README_BEFORE_M6.md**",
    "preserves an earlier README version."
]

lines = content.splitlines(keepends=True)

updated_lines = [
    line for line in lines
    if not any(item in line for item in lines_to_remove)
]

updated_content = "".join(updated_lines)

if updated_content != content:
    readme_path.write_text(updated_content, encoding="utf-8")
    print("PASS — Obsolete backup references removed")
else:
    print("No backup references found — README unchanged")

print("Saved to:", readme_path)

PASS — Obsolete backup references removed
Saved to: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Clear_Attack_Simulation\README.md



### M7 — Final ML Features & Ground Truth Validation

**Objective:** Verify the compatibility of the final ML feature dataset and Ground Truth before handing them over to the Machine Learning team.

**Validation Checks:**
1. Confirm that all required Parquet files exist.
2. Verify the presence and uniqueness of `(user, day)` keys.
3. Check the overlap between Final ML Features and Ground Truth.
4. Identify labeled and unlabeled feature rows.
5. Confirm that all attack-day labels are preserved.
6. Validate `is_attack_day` values and identify missing labels.
7. Compare overlapping feature keys with the approved Stage 3 dataset.
8. Detect potential label leakage in the ML feature columns.

**Expected Output:** A clear PASS/WARNING/FAIL report describing whether the datasets can be safely joined and handed over for ML development.

**Scope:** Read-only validation. No dataset modifications, injection, or feature recalculation.


In [8]:

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# ============================================================
# M7 — FINAL ML HANDOFF VALIDATION
# READ-ONLY
# ============================================================

root = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\Clear_Attack_Simulation"
)

paths = {
    "Final ML Features": root / "data/final/behavior_features_r62_ml_final.parquet",
    "User-Day Ground Truth": root / "data/ground_truth/ground_truth_user_days.parquet",
    "Stage 3 Features": root / "data/stages_output/behavior_features_injected_final.parquet"
}

results = []

# ------------------------------------------------------------
# 1. Verify files and schemas
# ------------------------------------------------------------

for name, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

    meta = pq.ParquetFile(path).metadata
    print(f"{name}: {meta.num_rows:,} rows × {meta.num_columns} columns")

# Load only necessary columns for key and label checks
final_schema = pq.read_schema(paths["Final ML Features"]).names
gt_schema = pq.read_schema(paths["User-Day Ground Truth"]).names
stage_schema = pq.read_schema(paths["Stage 3 Features"]).names

required_keys = {"user", "day"}

for name, columns in [
    ("Final ML Features", final_schema),
    ("Ground Truth", gt_schema),
    ("Stage 3 Features", stage_schema)
]:
    missing = required_keys - set(columns)
    if missing:
        raise ValueError(f"{name} missing keys: {missing}")

if "is_attack_day" not in gt_schema:
    raise ValueError("Ground Truth missing is_attack_day")

features = pd.read_parquet(
    paths["Final ML Features"],
    columns=["user", "day"]
)

gt = pd.read_parquet(
    paths["User-Day Ground Truth"],
    columns=["user", "day", "is_attack_day"]
)

stage = pd.read_parquet(
    paths["Stage 3 Features"],
    columns=["user", "day"]
)

# ------------------------------------------------------------
# 2. Key integrity
# ------------------------------------------------------------

def key_report(df, name):
    missing = int(df[["user", "day"]].isna().any(axis=1).sum())
    duplicates = int(df.duplicated(["user", "day"]).sum())

    print(f"\n{name}")
    print(f"Missing keys: {missing:,}")
    print(f"Duplicate keys: {duplicates:,}")

    return missing == 0 and duplicates == 0

final_keys_ok = key_report(features, "Final ML Features")
gt_keys_ok = key_report(gt, "Ground Truth")
stage_keys_ok = key_report(stage, "Stage 3 Features")

# ------------------------------------------------------------
# 3. Label validation
# ------------------------------------------------------------

labels_valid = bool(
    gt["is_attack_day"].notna().all()
    and gt["is_attack_day"].isin([0, 1]).all()
)

attack_days = int(gt["is_attack_day"].eq(1).sum())
benign_labeled_days = int(gt["is_attack_day"].eq(0).sum())

print("\nGROUND TRUTH LABELS")
print("Valid binary labels:", labels_valid)
print("Attack days:", f"{attack_days:,}")
print("No-injection days:", f"{benign_labeled_days:,}")

# ------------------------------------------------------------
# 4. Join coverage
# ------------------------------------------------------------

joined = features.merge(
    gt,
    on=["user", "day"],
    how="left",
    validate="one_to_one",
    indicator=True
)

matched = int(joined["_merge"].eq("both").sum())
unmatched_features = int(joined["_merge"].eq("left_only").sum())

gt_coverage = gt.merge(
    features,
    on=["user", "day"],
    how="left",
    validate="one_to_one",
    indicator=True
)

missing_gt = int(gt_coverage["_merge"].eq("left_only").sum())

matched_attacks = int(
    joined.loc[
        joined["_merge"].eq("both"),
        "is_attack_day"
    ].eq(1).sum()
)

print("\nFINAL ML JOIN COVERAGE")
print("Matched feature rows:", f"{matched:,}")
print("Unlabeled feature rows:", f"{unmatched_features:,}")
print("Ground Truth rows missing from Final Features:", f"{missing_gt:,}")
print("Attack labels preserved:", f"{matched_attacks:,}/{attack_days:,}")

# ------------------------------------------------------------
# 5. Compare Stage 3 key coverage
# ------------------------------------------------------------

stage_comparison = stage.merge(
    gt[["user", "day"]],
    on=["user", "day"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

stage_matches = bool(
    stage_comparison["_merge"].eq("both").all()
)

print("\nSTAGE 3 REFERENCE")
print("Stage 3 keys match Ground Truth:", stage_matches)

# ------------------------------------------------------------
# 6. Potential target leakage
# ------------------------------------------------------------

leakage_columns = {
    "is_attack_day",
    "day_status",
    "scenario_id",
    "stealth_level",
    "injected_email",
    "injected_usb",
    "injected_http",
    "injected_events_total",
    "injected_channel_count",
    "planned_channels",
    "is_scheduled",
    "injection_formula",
    "ground_truth_source"
}

found_leakage = sorted(leakage_columns.intersection(final_schema))

print("\nPOTENTIAL LABEL LEAKAGE")
print("Ground Truth metadata columns in Final Features:", found_leakage)

# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

checks = {
    "All files exist": True,
    "Final feature keys valid": final_keys_ok,
    "Ground Truth keys valid": gt_keys_ok,
    "Stage 3 feature keys valid": stage_keys_ok,
    "Ground Truth binary labels valid": labels_valid,
    "Stage 3 keys match Ground Truth": stage_matches,
    "All attack labels preserved": matched_attacks == attack_days,
    "All GT keys present in Final Features": missing_gt == 0,
    "No known GT metadata in Final Features": len(found_leakage) == 0
}

print("\n" + "=" * 60)
print("FINAL ML HANDOFF VALIDATION SUMMARY")
print("=" * 60)

for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {check}")

if all(checks.values()):
    print("\nPASS — Structural handoff checks completed.")
    if unmatched_features:
        print(
            f"WARNING — {unmatched_features:,} final feature rows "
            "have no synthetic Ground Truth label."
        )
    print("Feature values and temporal leakage still require separate validation.")
else:
    print("\nREVIEW REQUIRED — Do not assume the datasets are ready.")

print("\nNo source files were modified.")


Final ML Features: 387,162 rows × 116 columns
User-Day Ground Truth: 69,634 rows × 13 columns
Stage 3 Features: 69,634 rows × 115 columns

Final ML Features
Missing keys: 0
Duplicate keys: 0

Ground Truth
Missing keys: 0
Duplicate keys: 0

Stage 3 Features
Missing keys: 0
Duplicate keys: 0

GROUND TRUTH LABELS
Valid binary labels: True
Attack days: 12,636
No-injection days: 56,998

FINAL ML JOIN COVERAGE
Matched feature rows: 69,634
Unlabeled feature rows: 317,528
Ground Truth rows missing from Final Features: 0
Attack labels preserved: 12,636/12,636

STAGE 3 REFERENCE
Stage 3 keys match Ground Truth: True

POTENTIAL LABEL LEAKAGE
Ground Truth metadata columns in Final Features: []

FINAL ML HANDOFF VALIDATION SUMMARY
PASS — All files exist
PASS — Final feature keys valid
PASS — Ground Truth keys valid
PASS — Stage 3 feature keys valid
PASS — Ground Truth binary labels valid
PASS — Stage 3 keys match Ground Truth
PASS — All attack labels preserved
PASS — All GT keys present in Final Fe


### M8 — Final ML Feature Value Consistency Validation

**Objective:** Verify that the final ML feature dataset preserves the approved Stage 3 feature values for all 69,634 matching user-day records.

**Validation Checks:**
1. Compare the column schemas of Stage 3 Features and Final ML Features.
2. Identify added, missing, and common columns.
3. Match records using the unique `(user, day)` key.
4. Compare all shared feature values, including missing values (NaN).
5. Check numerical differences using strict floating-point tolerances.
6. Report mismatched columns, affected rows, and maximum numerical differences.
7. Verify that the 30 engineered features are preserved.

**Expected Output:** A PASS/FAIL report showing whether the final ML dataset retains the validated Stage 3 feature values.

**Scope:** Read-only validation. No modifications to Parquet files, Ground Truth, or existing QA reports.


In [9]:

from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

# ============================================================
# M8 — FINAL ML FEATURE VALUE CONSISTENCY
# READ-ONLY
# ============================================================

root = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\Clear_Attack_Simulation"
)

stage_path = (
    root / "data/stages_output"
    / "behavior_features_injected_final.parquet"
)

final_path = (
    root / "data/final"
    / "behavior_features_r62_ml_final.parquet"
)

# Numerical comparison tolerances
RTOL = 1e-9
ATOL = 1e-10

# ------------------------------------------------------------
# 1. Read datasets
# ------------------------------------------------------------

stage = pd.read_parquet(stage_path)
final = pd.read_parquet(final_path)

keys = ["user", "day"]

for name, df in [("Stage 3", stage), ("Final ML", final)]:
    assert all(k in df.columns for k in keys), f"Missing keys in {name}"
    assert not df[keys].isna().any().any(), f"Null keys in {name}"
    assert not df.duplicated(keys).any(), f"Duplicate keys in {name}"

print("STAGE 3:", stage.shape)
print("FINAL ML:", final.shape)

# ------------------------------------------------------------
# 2. Schema comparison
# ------------------------------------------------------------

stage_cols = set(stage.columns)
final_cols = set(final.columns)

common_cols = [
    col for col in stage.columns if col in final_cols
]

missing_in_final = sorted(stage_cols - final_cols)
added_in_final = sorted(final_cols - stage_cols)

print("\nSCHEMA COMPARISON")
print("Common columns:", len(common_cols))
print("Missing in Final ML:", missing_in_final)
print("Additional in Final ML:", added_in_final)

# ------------------------------------------------------------
# 3. Match Stage 3 records with Final ML
# ------------------------------------------------------------

stage_idx = stage.set_index(keys).sort_index()
final_idx = final.set_index(keys).sort_index()

missing_keys = stage_idx.index.difference(final_idx.index)

if len(missing_keys):
    raise ValueError(
        f"{len(missing_keys)} Stage 3 records missing from Final ML"
    )

final_matched = final_idx.loc[stage_idx.index]

print("\nROW MATCHING")
print("Matched rows:", f"{len(final_matched):,}")
print("Missing Stage 3 rows:", len(missing_keys))

# ------------------------------------------------------------
# 4. Compare values across all shared features
# ------------------------------------------------------------

feature_cols = [
    col for col in stage_idx.columns
    if col in final_matched.columns
]

comparison_results = []
total_mismatch_cells = 0

for col in feature_cols:

    a = stage_idx[col]
    b = final_matched[col]

    null_mismatch = a.isna() != b.isna()

    if (
        pd.api.types.is_numeric_dtype(a)
        and pd.api.types.is_numeric_dtype(b)
    ):
        equal = np.isclose(
            a.to_numpy(dtype=float),
            b.to_numpy(dtype=float),
            rtol=RTOL,
            atol=ATOL,
            equal_nan=True
        )

        mismatches = ~equal

        valid = a.notna() & b.notna()

        if valid.any():
            max_diff = float(
                np.max(
                    np.abs(
                        a[valid].to_numpy(dtype=float)
                        - b[valid].to_numpy(dtype=float)
                    )
                )
            )
        else:
            max_diff = np.nan

    else:
        equal = a.eq(b).fillna(False) | (
            a.isna() & b.isna()
        )
        mismatches = (~equal).to_numpy()
        max_diff = np.nan

    count = int(np.sum(mismatches))
    total_mismatch_cells += count

    comparison_results.append({
        "column": col,
        "mismatched_rows": count,
        "null_mismatches": int(null_mismatch.sum()),
        "max_absolute_difference": max_diff
    })

report = pd.DataFrame(comparison_results)

# ------------------------------------------------------------
# 5. Report differences
# ------------------------------------------------------------

problem_columns = report[
    report["mismatched_rows"] > 0
].sort_values("mismatched_rows", ascending=False)

print("\nVALUE COMPARISON")
print("Compared feature columns:", len(feature_cols))
print("Compared feature cells:", f"{len(stage_idx) * len(feature_cols):,}")
print("Mismatched cells:", f"{total_mismatch_cells:,}")
print("Columns with mismatches:", len(problem_columns))

if len(problem_columns):
    display(problem_columns)
else:
    print("PASS — All shared feature values match.")

# ------------------------------------------------------------
# 6. Final decision
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("M8 — FINAL VALIDATION SUMMARY")
print("=" * 60)

schema_ok = len(missing_in_final) == 0
values_ok = total_mismatch_cells == 0
rows_ok = len(missing_keys) == 0

print(
    f"{'PASS' if schema_ok else 'FAIL'} — "
    "All Stage 3 columns preserved"
)
print(
    f"{'PASS' if rows_ok else 'FAIL'} — "
    "All Stage 3 user-day records preserved"
)
print(
    f"{'PASS' if values_ok else 'FAIL'} — "
    "Shared feature values consistent"
)

if schema_ok and rows_ok and values_ok:
    print("\nPASS — Final ML preserves all approved Stage 3 feature values.")
else:
    print("\nREVIEW REQUIRED — Differences detected.")

print("\nNo source files were modified.")


STAGE 3: (69634, 115)
FINAL ML: (387162, 116)

SCHEMA COMPARISON
Common columns: 115
Missing in Final ML: []
Additional in Final ML: ['y']

ROW MATCHING
Matched rows: 69,634
Missing Stage 3 rows: 0

VALUE COMPARISON
Compared feature columns: 113
Compared feature cells: 7,868,642
Mismatched cells: 0
Columns with mismatches: 0
PASS — All shared feature values match.

M8 — FINAL VALIDATION SUMMARY
PASS — All Stage 3 columns preserved
PASS — All Stage 3 user-day records preserved
PASS — Shared feature values consistent

PASS — Final ML preserves all approved Stage 3 feature values.

No source files were modified.
